In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

def summarize_df(df, name_field=None):
    # Basic counts
    row_count = int(df.shape[0])
    # Ensure timestamp column exists and is numeric
    if 'timestamp' in df.columns and row_count > 0:
        ts_min = int(df['timestamp'].min())
        ts_max = int(df['timestamp'].max())
        # convert to timezone-aware ISO strings using UTC then convert to Asia/Shanghai
        ts_min_iso = pd.to_datetime(ts_min, unit='s', utc=True).tz_convert(tz).isoformat()
        ts_max_iso = pd.to_datetime(ts_max, unit='s', utc=True).tz_convert(tz).isoformat()
        earliest = {'unix': ts_min, 'iso_Asia/Shanghai': ts_min_iso}
        latest = {'unix': ts_max, 'iso_Asia/Shanghai': ts_max_iso}
    else:
        earliest = None
        latest = None

    # cmdb_id summary
    if 'cmdb_id' in df.columns:
        unique_cmdb = list(pd.Series(df['cmdb_id'].unique())[:50])
        cmdb_count = int(pd.Series(df['cmdb_id'].nunique()))
    else:
        unique_cmdb = []
        cmdb_count = 0

    # distinct name field summary (kpi_name / trace_name / log_name) if provided
    distinct_names = None
    if name_field and name_field in df.columns:
        distinct_names = list(pd.Series(df[name_field].unique())[:50])

    # top 5 rows (only existing CSV columns). Convert to plain Python types via to_dict.
    top_rows = df.head(5).to_dict(orient='records')

    summary = {
        'row_count': row_count,
        'earliest_timestamp': earliest,
        'latest_timestamp': latest,
        'distinct_cmdb_count': cmdb_count,
        'cmdb_ids_sample_up_to_50': unique_cmdb,
        f'distinct_{name_field}_up_to_50' if name_field else 'distinct_names': distinct_names,
        'top_5_rows': top_rows
    }
    return summary

metric_summary = summarize_df(metric_df, name_field='kpi_name')
trace_summary = summarize_df(trace_df, name_field='trace_name')
log_summary = summarize_df(log_df, name_field='log_name')
error_logs_summary = summarize_df(error_logs_df, name_field=None)

summary = {
    'metric.csv': metric_summary,
    'trace.csv': trace_summary,
    'log.csv': log_summary,
    'error_logs.csv': error_logs_summary
}

summary

```
Out[1]:
```
```python
# Plain-English summary based on the previous execution results (displayed as a single string)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def to_iso(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

# Reuse loaded DataFrames from the kernel: metric_df, trace_df, log_df, error_logs_df
# Build concise summaries
m_rows = metric_df.shape[0]
m_min, m_max = int(metric_df['timestamp'].min()), int(metric_df['timestamp'].max())
m_cmdb_sample = list(pd.Series(metric_df['cmdb_id'].unique())[:12])
m_kpis_sample = list(pd.Series(metric_df['kpi_name'].unique())[:10])
m_top5_lines = [
    f"{r['timestamp']} | {r['cmdb_id']} | {r['kpi_name']} = {r['value']}"
    for r in metric_df.head(5).to_dict(orient='records')
]

t_rows = trace_df.shape[0]
t_min, t_max = int(trace_df['timestamp'].min()), int(trace_df['timestamp'].max())
t_cmdb_sample = list(pd.Series(trace_df['cmdb_id'].unique())[:12])
t_traces_sample = list(pd.Series(trace_df['trace_name'].unique())[:10])
t_top5_lines = [
    f"{r['timestamp']} | {r['cmdb_id']} | {r['trace_name']} = {r['value']}"
    for r in trace_df.head(5).to_dict(orient='records')
]

l_rows = log_df.shape[0]
l_min, l_max = int(log_df['timestamp'].min()), int(log_df['timestamp'].max())
l_cmdb_sample = list(pd.Series(log_df['cmdb_id'].unique())[:12])
l_logs_sample = list(pd.Series(log_df['log_name'].unique())[:10])
l_top5_lines = [
    f"{r['timestamp']} | {r['cmdb_id']} | {r['log_name']} = {r['value']}"
    for r in log_df.head(5).to_dict(orient='records')
]

e_rows = error_logs_df.shape[0]

summary_text = (
    "Summary of telemetry CSV files (times shown in Asia/Shanghai UTC+8):\n\n"
    f"metric.csv:\n"
    f"- Total rows: {m_rows}\n"
    f"- Time range: {to_iso(m_min)} to {to_iso(m_max)}\n"
    f"- Distinct cmdb_id count: {int(metric_df['cmdb_id'].nunique())}; sample (up to 12): {m_cmdb_sample}\n"
    f"- Distinct kpi_name sample (up to 10): {m_kpis_sample}\n"
    f"- Top 5 rows (timestamp | cmdb_id | kpi_name = value):\n  " + "\n  ".join(m_top5_lines) + "\n\n"
    f"trace.csv:\n"
    f"- Total rows: {t_rows}\n"
    f"- Time range: {to_iso(t_min)} to {to_iso(t_max)}\n"
    f"- Distinct cmdb_id count: {int(trace_df['cmdb_id'].nunique())}; sample (up to 12): {t_cmdb_sample}\n"
    f"- Distinct trace_name sample (up to 10): {t_traces_sample}\n"
    f"- Top 5 rows (timestamp | cmdb_id | trace_name = value):\n  " + "\n  ".join(t_top5_lines) + "\n\n"
    f"log.csv:\n"
    f"- Total rows: {l_rows}\n"
    f"- Time range: {to_iso(l_min)} to {to_iso(l_max)}\n"
    f"- Distinct cmdb_id count: {int(log_df['cmdb_id'].nunique())}; sample (up to 12): {l_cmdb_sample}\n"
    f"- Distinct log_name values: {l_logs_sample}\n"
    f"- Top 5 rows (timestamp | cmdb_id | log_name = value):\n  " + "\n  ".join(l_top5_lines) + "\n\n"
    f"error_logs.csv:\n"
    f"- Total rows: {e_rows}\n"
    f"- No timestamps or entries (empty file)\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'row_count': 325376, 'earliest_timestamp': {'unix': 1647842400, 'iso_Asia/Shanghai': '2022-03-21T14:00:00+08:00'}, 'latest_timestamp': {'unix': 1647846000, 'iso_Asia/Shanghai': '2022-03-21T15:00:00+08:00'}, 'distinct_cmdb_count': 59, 'cmdb_ids_sample_up_to_50': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'distinct_kpi_name_up_to_50': ['app.grpc.count', 'app.grpc.mrt', 'app.grpc.rr', 'app.grpc.sr', 'app.http.count', 'app.http.mrt', 'app.http.rr', 'app.http.sr', 'runtime.java_lang_ClassLoading_LoadedClassCount', 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'runtime.java_lang_ClassLoading_Verbose', 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'runtime.java_lang_Compilation_TotalCompilationTime', 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_endTime.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_id.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_committed.Tenured_Gen.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Code_Cache.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Compressed_Class_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Eden_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Metaspace.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Survivor_Space.MarkSweepCompact', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.Copy', 'runtime.java_lang_GarbageCollector_LastGcInfo_memoryUsageAfterGc_init.Tenured_Gen.MarkSweepCompact'], 'top_5_rows': [{'timestamp': 1647842400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'value': 249.0}, {'timestamp': 1647842400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'value': 0.0026718875502028}, {'timestamp': 1647842400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'value': 100.0}, {'timestamp': 1647842400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'value': 100.0}, {'timestamp': 1647842400, 'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'value': 24.0}]}, 'trace.csv': {'row_count': 36312, 'earliest_timestamp': {'unix': 1647844200, 'iso_Asia/Shanghai': '2022-03-21T14:30:00+08:00'}, 'latest_timestamp': {'unix': 1647845940, 'iso_Asia/Shanghai': '2022-03-21T14:59:00+08:00'}, 'distinct_cmdb_count': 40, 'cmdb_ids_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0'], 'distinct_trace_name_up_to_50': ['trace.from_frontend-0.duration_mean', 'trace.from_frontend-0.duration_p95', 'trace.from_frontend-0.error_rate', 'trace.from_frontend-0.row_count', 'trace.from_frontend-1.duration_mean', 'trace.from_frontend-1.duration_p95', 'trace.from_frontend-1.error_rate', 'trace.from_frontend-1.row_count', 'trace.from_frontend-2.duration_mean', 'trace.from_frontend-2.duration_p95', 'trace.from_frontend-2.error_rate', 'trace.from_frontend-2.row_count', 'trace.from_frontend2-0.duration_mean', 'trace.from_frontend2-0.duration_p95', 'trace.from_frontend2-0.error_rate', 'trace.from_frontend2-0.row_count', 'trace.from_checkoutservice-0.duration_mean', 'trace.from_checkoutservice-0.duration_p95', 'trace.from_checkoutservice-0.error_rate', 'trace.from_checkoutservice-0.row_count', 'trace.from_checkoutservice-1.duration_mean', 'trace.from_checkoutservice-1.duration_p95', 'trace.from_checkoutservice-1.error_rate', 'trace.from_checkoutservice-1.row_count', 'trace.from_checkoutservice-2.duration_mean', 'trace.from_checkoutservice-2.duration_p95', 'trace.from_checkoutservice-2.error_rate', 'trace.from_checkoutservice-2.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.error_rate', 'trace.self.row_count', 'trace.from_checkoutservice2-0.duration_mean', 'trace.from_checkoutservice2-0.duration_p95', 'trace.from_checkoutservice2-0.error_rate', 'trace.from_checkoutservice2-0.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.to_cartservice-0.duration_mean', 'trace.to_cartservice-0.duration_p95', 'trace.to_cartservice-0.error_rate', 'trace.to_cartservice-0.row_count', 'trace.to_cartservice-2.duration_mean', 'trace.to_cartservice-2.duration_p95', 'trace.to_cartservice-2.error_rate', 'trace.to_cartservice-2.row_count', 'trace.to_currencyservice-0.duration_mean', 'trace.to_currencyservice-0.duration_p95'], 'top_5_rows': [{'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_mean', 'value': 1.7e-05}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_p95', 'value': 2.3e-05}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.error_rate', 'value': 0.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'value': 17.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-1.duration_mean', 'value': 1.7e-05}]}, 'log.csv': {'row_count': 1842, 'earliest_timestamp': {'unix': 1647844200, 'iso_Asia/Shanghai': '2022-03-21T14:30:00+08:00'}, 'latest_timestamp': {'unix': 1647846000, 'iso_Asia/Shanghai': '2022-03-21T15:00:00+08:00'}, 'distinct_cmdb_count': 31, 'cmdb_ids_sample_up_to_50': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'redis-cart-0'], 'distinct_log_name_up_to_50': ['log.error_count', 'log.row_count'], 'top_5_rows': [{'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-0', 'log_name': 'log.row_count', 'value': 122.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-1', 'log_name': 'log.row_count', 'value': 112.0}, {'timestamp': 1647844200, 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': 0.0}]}, 'error_logs.csv': {'row_count': 0, 'earliest_timestamp': None, 'latest_timestamp': None, 'distinct_cmdb_count': 0, 'cmdb_ids_sample_up_to_50': [], 'distinct_names': None, 'top_5_rows': []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel
# Ensure correct dtypes
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95, P5 and total_points per (cmdb_id, kpi_name) using full series
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(total_points='count',
         global_P95=lambda x: x.quantile(0.95),
         global_P5=lambda x: x.quantile(0.05))
)

# Keep thresholds in kernel for inspection if needed
thresholds_df = thresholds.copy()

# 2) Filter metric rows to the incident window
start_ts = 1647844200
end_ts = 1647846000
metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Within window compute anomaly_count, earliest_anomaly_ts, max/min in window
# Merge thresholds onto window rows to evaluate anomalies using global thresholds
mw = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left', suffixes=('','_thr'))

# Mark anomalies per rule: value >= global_P95 OR value <= global_P5
mw['is_anomaly'] = (mw['value'] >= mw['global_P95']) | (mw['value'] <= mw['global_P5'])

# Aggregate per pair within window
window_agg = (
    mw
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        points_in_window=('value','count'),
        anomaly_count=('is_anomaly','sum'),
        earliest_anomaly_ts=('timestamp', lambda s: int(s.loc[mw.loc[s.index,'is_anomaly']].min()) if s.loc[mw.loc[s.index,'is_anomaly']].size>0 else None),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
    )
)

# Note: the lambda above uses index alignment to get earliest timestamp where is_anomaly True.
# However groupby with lambda referencing mw via indices can be fragile; compute safer way below if any None issues.

# Safer calculation for earliest_anomaly_ts:
anomaly_rows = mw[mw['is_anomaly']].copy()
earliest_ts_series = anomaly_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
# Merge earliest timestamps into window_agg
window_agg = window_agg.drop(columns=['earliest_anomaly_ts']).merge(earliest_ts_series, on=['cmdb_id','kpi_name'], how='left')
# Convert NaN to None and to int where present
window_agg['earliest_anomaly_ts'] = window_agg['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) else None)

# 4) Combine with global thresholds and compute severity for pairs with anomaly_count>0
combined = window_agg.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

def compute_severity(row):
    gp95 = row['global_P95']
    gp5 = row['global_P5']
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    # If both anomalies present, prefer high-side severity first as per spec
    if pd.notna(maxv) and pd.notna(gp95) and (maxv > gp95):
        if gp95 == 0:
            return float('inf')
        return float(maxv / gp95)
    if pd.notna(minv) and pd.notna(gp5) and (minv < gp5):
        if minv == 0:
            return float('inf')
        return float(gp5 / minv)
    return 1.0

combined['severity'] = combined.apply(lambda r: compute_severity(r) if r['anomaly_count']>0 else 0.0, axis=1)

# 5) Prepare final compact DataFrame: filter anomaly_count>0, select columns, sort, limit top 20
result_df = (
    combined[combined['anomaly_count']>0]
    .loc[:, ['cmdb_id','kpi_name','total_points','global_P95','global_P5',
             'anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window','severity']]
    .sort_values(by=['anomaly_count','severity'], ascending=[False, False])
    .reset_index(drop=True)
)

# Keep full intermediate tables in kernel variables (thresholds_df, metric_window, mw, window_agg, combined)
# Provide compact outputs: top 20 rows, total distinct pairs processed, number of pairs with anomalies in window
total_pairs = int(thresholds_df.shape[0])
anomalous_pairs_count = int(result_df.shape[0])

# Limit result display to top 20 rows as requested
result_df_top20 = result_df.head(20)

# Display compact outputs (DataFrame + counts)
result_df_top20, total_pairs, anomalous_pairs_count

```
Out[2]:
```
Summary (plain English)

- Scope: I processed 5,346 distinct (cmdb_id, kpi_name) pairs from metric.csv using the full historical series to compute global P95 and P5, then evaluated anomalies inside the incident window 1647844200–1647846000 (2022-03-21 14:30:00 to 15:00:00 Asia/Shanghai).

- Counts:
  - Total distinct (cmdb_id, kpi_name) pairs processed: 5,346.
  - Pairs with anomaly_count > 0 inside the window: 5,301.

- Key findings from the top results (top 20 by anomaly_count, then severity):
  - Many top entries are for adservice (and adservice-0/1/2) and container/node-related KPIs.
  - For the shown top rows: total_points = 61 (per pair), anomaly_count = 31, and earliest anomaly timestamp ≈ 1647844200 (about 2022-03-21 14:30:00+08).
  - A common pattern: global_P95 and global_P5 are both 0.0 for many KPIs, while the window contains non-zero max values and zero minima. That leads to severity being computed as infinite (division by zero) for those pairs.
  - Example columns observed in top rows: cmdb_id, kpi_name, total_points (61), global_P95 (0.0), global_P5 (0.0), anomaly_count (31), earliest_anomaly_ts (~1647844200), max_value_in_window (varies, >0), min_value_in_window (0.0), severity (inf where division by zero occurred).

- Interpretation / note:
  - The high number of pairs flagged (5,301 of 5,346) indicates many KPIs fall outside their global 5th–95th percentile bounds during the window, but many of these are driven by global thresholds equal to zero. Where global_P95 or global_P5 equals zero, the computed severity becomes infinite and should be treated carefully (these likely indicate sparse or heavily skewed metrics rather than classical extreme outliers).
  - Recommend reviewing KPIs with zero global thresholds and verifying whether those metrics are meaningful to include in automated severity ranking (or require thresholding/regularization to avoid division-by-zero artifacts).

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  total_points  global_P95  global_P5  anomaly_count  earliest_anomaly_ts  max_value_in_window  min_value_in_window  severity
0     adservice  runtime.java_lang_Memory_ObjectPendingFinaliza...            61         0.0        0.0             31         1.647844e+09             0.083333                  0.0       inf
1     adservice                         runtime.jvm_classes_loaded            61      5162.0     5162.0             31         1.647844e+09          5162.000000                  0.0       inf
2   adservice-0  container.node-6.container_cpu_cfs_throttled_p...            61         0.0        0.0             31         1.647844e+09            15.000000                  0.0       inf
3   adservice-0  container.node-6.container_cpu_cfs_throttled_s...            61         0.0        0.0             31         1.647844e+09             1.800396                  0.0       inf
4   adservice-0     container.node-6.container_fs_inodes./dev/vda1            61         0.0        0.0             31         1.647844e+09             1.500000                  0.0       inf
5   adservice-0       container.node-6.container_fs_reads./dev/vda            61         0.0        0.0             31         1.647844e+09          2894.000000                  0.0       inf
6   adservice-0    container.node-6.container_fs_reads_MB./dev/vda            61         0.0        0.0             31         1.647844e+09          2299.878906                  0.0       inf
7   adservice-0      container.node-6.container_fs_writes./dev/vda            61         0.0        0.0             31         1.647844e+09          1749.000000                  0.0       inf
8   adservice-0   container.node-6.container_fs_writes_MB./dev/vda            61         0.0        0.0             31         1.647844e+09           596.621094                  0.0       inf
9   adservice-1  container.node-6.container_cpu_cfs_throttled_p...            61         0.0        0.0             31         1.647844e+09            20.000000                  0.0       inf
10  adservice-1  container.node-6.container_cpu_cfs_throttled_s...            61         0.0        0.0             31         1.647844e+09             2.838158                  0.0       inf
11  adservice-1     container.node-6.container_fs_inodes./dev/vda1            61         0.0        0.0             31         1.647844e+09             0.500000                  0.0       inf
12  adservice-1       container.node-6.container_fs_reads./dev/vda            61         0.0        0.0             31         1.647844e+09          3858.500000                  0.0       inf
13  adservice-1    container.node-6.container_fs_reads_MB./dev/vda            61         0.0        0.0             31         1.647844e+09          2739.710938                  0.0       inf
14  adservice-1      container.node-6.container_fs_writes./dev/vda            61         0.0        0.0             31         1.647844e+09          1762.000000                  0.0       inf
15  adservice-1   container.node-6.container_fs_writes_MB./dev/vda            61         0.0        0.0             31         1.647844e+09           596.617188                  0.0       inf
16  adservice-1  container.node-6.container_memory_failures.con...            61         0.0        0.0             31         1.647844e+09            33.000000                  0.0       inf
17  adservice-1  container.node-6.container_memory_failures.hie...            61         0.0        0.0             31         1.647844e+09            33.000000                  0.0       inf
18  adservice-2  container.node-5.container_cpu_cfs_throttled_p...            61         0.0        0.0             31         1.647844e+09            18.000000                  0.0       inf
19  adservice-2  container.node-5.container_cpu_cfs_throttled_s...            61         0.0        0.0             31         1.647844e+09             2.004405                  0.0       inf, 5346, 5301)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse 'combined' DataFrame from previous run which contains per-(cmdb_id,kpi_name) aggregates,
# including total_points, global_P95, global_P5, points_in_window, anomaly_count, earliest_anomaly_ts, max/min in window, severity.

# If 'combined' not present, raise informative error (but per instructions it's already in kernel).
try:
    combined.shape
except NameError:
    raise RuntimeError("Expected variable 'combined' not found in kernel. Please run previous analysis step.")

# 1) Filter KPI names by resource-related substrings (case-insensitive)
substrs = ['cpu', 'memory', 'fs_', 'fs', 'fsreads', 'fswrites', 'reads', 'writes', 'io', 'disk',
           'throttl', 'network', 'pkt', 'latency', 'cfs', 'inodes']

# Prepare a lowercase kpi_name column for matching
combined['kpi_name_lc'] = combined['kpi_name'].str.lower().fillna('')

mask_resource = combined['kpi_name_lc'].apply(lambda s: any(sub in s for sub in substrs))

resource_pairs = combined[mask_resource].copy()

# 2) Exclude pairs where both global_P95 == 0.0 AND global_P5 == 0.0
resource_pairs = resource_pairs[~((resource_pairs['global_P95'] == 0.0) & (resource_pairs['global_P5'] == 0.0))].copy()

# Also restrict to those that have data in the incident window (points_in_window > 0)
# (combined was computed from window_agg so points_in_window exists)
resource_pairs = resource_pairs[resource_pairs['points_in_window'].fillna(0) > 0].copy()

# Counts requested
resource_pairs_count = int(resource_pairs.shape[0])
resource_anomalous_pairs_count = int(resource_pairs[resource_pairs['anomaly_count'] > 0].shape[0])

# 3A) Top 20 rows sorted by severity desc then anomaly_count desc, include specified columns.
# Only include pairs with anomaly_count > 0 as these are meaningful for severity sorting.
res_anomalous = resource_pairs[resource_pairs['anomaly_count'] > 0].copy()

# Convert earliest_anomaly_ts to ISO Asia/Shanghai for display (keep original as int in another col if needed)
def ts_to_iso_opt(ts):
    if pd.isna(ts) or ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

res_anomalous['earliest_anomaly_iso'] = res_anomalous['earliest_anomaly_ts'].apply(ts_to_iso_opt)

# Select and sort
res_top20 = (
    res_anomalous
    .loc[:, ['cmdb_id','kpi_name','total_points','global_P95','global_P5',
             'anomaly_count','earliest_anomaly_iso','max_value_in_window','min_value_in_window','severity']]
    .rename(columns={'earliest_anomaly_iso':'earliest_anomaly_ts'})
    .sort_values(by=['severity','anomaly_count'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# 3B) Aggregated per-cmdb_id summary (top 20) sorted by number_of_anomalous_kpis desc then max_severity desc.
agg = (
    res_anomalous
    .groupby('cmdb_id', as_index=False)
    .agg(
        number_of_anomalous_kpis=('kpi_name','nunique'),
        max_severity=('severity','max'),
        earliest_anomaly_ts_overall=('earliest_anomaly_ts','min')
    )
)

# Convert earliest_anomaly_ts_overall to ISO
agg['earliest_anomaly_ts_overall'] = agg['earliest_anomaly_ts_overall'].apply(ts_to_iso_opt)

agg_top20 = (
    agg
    .sort_values(by=['number_of_anomalous_kpis','max_severity'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
    .loc[:, ['cmdb_id','number_of_anomalous_kpis','max_severity','earliest_anomaly_ts_overall']]
)

# Clean up temporary helper column
# (Keep combined intact except remove helper kpi_name_lc)
combined.drop(columns=['kpi_name_lc'], inplace=True, errors=False)

# Final compact outputs: res_top20 (per-pair), agg_top20 (per-cmdb_id), counts
res_top20, agg_top20, resource_pairs_count, resource_anomalous_pairs_count

```
Out[3]:
```
Summary (plain English, incident window 2022-03-21 14:30:00 to 15:00:00 Asia/Shanghai):

- Filtering metric KPIs for resource-related names and excluding pairs where both global P95 and P5 are zero yielded 2,448 (cmdb_id, kpi_name) pairs. Of those, 2,407 pairs had anomaly_count > 0 inside the incident window.

- Top per-(cmdb_id, kpi_name) examples (from the top-20 by severity then anomaly_count):
  - checkoutservice-0 · container_cpu_cfs_throttled... — total_points=61, global_P95=4.0, global_P5=1.0, anomaly_count=6, earliest_anomaly=2022-03-21T14:38:00+08:00, max_in_window=4.0, min_in_window=0.0, severity=inf.
  - adservice-2 · container_fs_reads_MB./dev/vda — total_points=61, global_P95≈0.6406, global_P5=0.0, anomaly_count=29, earliest_anomaly=2022-03-21T14:30:00+08:00, max_in_window≈12,261.9, severity≈19140.
  - node-5 · system.io.rkb_s — total_points=61, global_P95=12.0, global_P5=0.0, anomaly_count=29, earliest_anomaly=2022-03-21T14:30:00+08:00, max_in_window≈122,872.0, severity≈10239.
  - Multiple adservice* entries show high anomaly counts and very large severities driven by large max values vs. global P95 or by zero/near-zero thresholds.

- Top components (aggregated per cmdb_id, top 20 by number_of_anomalous_kpis then max_severity):
  - adservice: 237 anomalous resource KPIs, max_severity ≈ 317.23, earliest anomaly 2022-03-21T14:30:00+08:00
  - adservice2: 237 anomalous KPIs, max_severity ≈ 145.40, earliest 14:30
  - frontend2-0, frontend-1, frontend-2, frontend-0: many anomalous resource KPIs (70–77 range for frontends)
  - checkoutservice-0: 70 anomalous KPIs and max_severity = inf (indicates division-by-zero or zero baseline in global threshold)
  - Other affected services: productcatalogservice*, shippingservice, cartservice, currencyservice, etc.
  - Several node-level metrics (node-5) appear in the per-pair list with very high IO-related severities.

- Interpretation and caveats:
  - Most anomalies are concentrated on IO, read/write, memory-failure, and CPU-throttling KPIs (names matching 'cpu', 'memory', 'fs', 'reads', 'writes', 'io', 'throttl', 'inodes', etc.).
  - Many severity values are extremely large or infinite. This is often due to very small or zero global thresholds (global_P95 or global_P5 near 0), so severity = max/global_P95 or gp5/min can blow up. Infinite severity flags should be treated as artifacts of zero baselines and investigated separately.
  - Earliest anomalous activity for many top items started right at the window start (2022-03-21T14:30:00+08:00).

- Recommended next steps:
  - Prioritize investigation of adservice/adservice2, checkoutservice*, and node-5 for IO, memory-failure and CPU-throttling signals.
  - Inspect raw metric time series and metric definitions for KPIs with zero or near-zero global thresholds to avoid misleading infinite severities (consider small regularization or minimum threshold before computing severity).
  - Correlate with traces/logs and node-level metrics for root-cause validation.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                           kpi_name  total_points    global_P95     global_P5  anomaly_count        earliest_anomaly_ts  max_value_in_window  min_value_in_window      severity
0   checkoutservice-0  container.node-6.container_cpu_cfs_throttled_p...            61  4.000000e+00  1.000000e+00              6  2022-03-21T14:38:00+08:00         4.000000e+00         0.000000e+00           inf
1         adservice-2    container.node-5.container_fs_reads_MB./dev/vda            61  6.406250e-01  0.000000e+00             29  2022-03-21T14:30:00+08:00         1.226190e+04         0.000000e+00  1.914053e+04
2              node-5                                    system.io.rkb_s            61  1.200000e+01  0.000000e+00             29  2022-03-21T14:30:00+08:00         1.228720e+05         0.000000e+00  1.023933e+04
3         adservice-2  container.node-5.container_memory_failures.con...            61  6.600000e+01  0.000000e+00              9  2022-03-21T14:33:00+08:00         9.482000e+04         0.000000e+00  1.436667e+03
4         adservice-2  container.node-5.container_memory_failures.hie...            61  6.600000e+01  0.000000e+00              9  2022-03-21T14:33:00+08:00         9.482000e+04         0.000000e+00  1.436667e+03
5         adservice-1  container.node-6.container_memory_failures.con...            61  4.950000e+01  0.000000e+00             19  2022-03-21T14:31:00+08:00         5.968050e+04         0.000000e+00  1.205667e+03
6         adservice-1  container.node-6.container_memory_failures.hie...            61  4.950000e+01  0.000000e+00             19  2022-03-21T14:31:00+08:00         5.968050e+04         0.000000e+00  1.205667e+03
7              node-5                                 system.io.avg_q_sz            61  1.000000e-02  0.000000e+00             31  2022-03-21T14:30:00+08:00         1.140000e+01         0.000000e+00  1.140000e+03
8         adservice-2       container.node-5.container_fs_reads./dev/vda            61  3.650000e+01  0.000000e+00             29  2022-03-21T14:30:00+08:00         3.700567e+04         0.000000e+00  1.013854e+03
9         adservice-0  container.node-6.container_memory_failures.con...            61  1.155000e+02  0.000000e+00              7  2022-03-21T14:44:00+08:00         4.818000e+04         0.000000e+00  4.171429e+02
10        adservice-0  container.node-6.container_memory_failures.hie...            61  1.155000e+02  0.000000e+00              7  2022-03-21T14:44:00+08:00         4.818000e+04         0.000000e+00  4.171429e+02
11          adservice  runtime.java_lang_OperatingSystem_FreePhysical...            61  3.681853e+08  3.680457e+08              7  2022-03-21T14:32:00+08:00         3.681853e+08         1.160192e+06  3.172283e+02
12       adservice2-0  container.node-6.container_memory_failures.con...            61  1.650000e+02  1.650000e+01              7  2022-03-21T14:30:00+08:00         4.156350e+04         0.000000e+00  2.519000e+02
13       adservice2-0  container.node-6.container_memory_failures.hie...            61  1.650000e+02  1.650000e+01              7  2022-03-21T14:30:00+08:00         4.156350e+04         0.000000e+00  2.519000e+02
14             node-5                                      system.io.r_s            61  1.500000e+00  0.000000e+00             30  2022-03-21T14:30:00+08:00         3.610000e+02         0.000000e+00  2.406667e+02
15         adservice2  runtime.java_lang_OperatingSystem_FreePhysical...            61  7.163221e+07  7.092156e+07              2  2022-03-21T14:59:00+08:00         7.116800e+07         4.877653e+05  1.454010e+02
16             node-5                                  system.cpu.iowait            61  6.000000e-02  0.000000e+00             20  2022-03-21T14:30:00+08:00         5.690000e+00         0.000000e+00  9.483333e+01
17             node-5                                     system.io.util            61  7.500000e-01  0.000000e+00             23  2022-03-21T14:30:00+08:00         4.940000e+01         0.000000e+00  6.586667e+01
18        adservice-2            container.node-5.container_memory_cache            61  8.994816e+06  7.913472e+06             31  2022-03-21T14:30:00+08:00         4.318740e+08         8.994816e+06  4.801366e+01
19       adservice2-0  container.node-6.container_cpu_cfs_throttled_s...            61  1.484456e-01  0.000000e+00             30  2022-03-21T14:30:00+08:00         6.415431e+00         0.000000e+00  4.321737e+01,                      cmdb_id  number_of_anomalous_kpis  max_severity earliest_anomaly_ts_overall
0                  adservice                       237    317.228302   2022-03-21T14:30:00+08:00
1                 adservice2                       237    145.400980   2022-03-21T14:30:00+08:00
2                frontend2-0                        77      2.095879   2022-03-21T14:30:00+08:00
3                 frontend-1                        77      1.768657   2022-03-21T14:30:00+08:00
4                 frontend-2                        76      1.692308   2022-03-21T14:30:00+08:00
5                 frontend-0                        74      1.538270   2022-03-21T14:30:00+08:00
6          checkoutservice-0                        70           inf   2022-03-21T14:30:00+08:00
7          checkoutservice-2                        70     10.385276   2022-03-21T14:30:00+08:00
8          checkoutservice-1                        70      2.000000   2022-03-21T14:30:00+08:00
9         checkoutservice2-0                        68      9.166667   2022-03-21T14:30:00+08:00
10   productcatalogservice-1                        54      1.987085   2022-03-21T14:30:00+08:00
11   productcatalogservice-2                        54      1.333333   2022-03-21T14:30:00+08:00
12   productcatalogservice-0                        53      1.522936   2022-03-21T14:30:00+08:00
13  productcatalogservice2-0                        45      3.000000   2022-03-21T14:30:00+08:00
14              adservice2-0                        40    251.900000   2022-03-21T14:30:00+08:00
15         shippingservice-2                        40     13.903314   2022-03-21T14:30:00+08:00
16             cartservice-1                        40      7.626924   2022-03-21T14:30:00+08:00
17         currencyservice-0                        39      2.633114   2022-03-21T14:30:00+08:00
18               adservice-0                        38    417.142857   2022-03-21T14:30:00+08:00
19         shippingservice-1                        38     13.543136   2022-03-21T14:30:00+08:00, 2448, 2407)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse variables from kernel: agg (per-cmdb_id aggregate), metric_window (metrics in window),
# resource_pairs (filtered resource-related pairs with thresholds), and thresholds if needed.

# 1) Top 10 components by number_of_anomalous_kpis from the stored aggregate 'agg'
top_components = agg.sort_values(by=['number_of_anomalous_kpis','max_severity'], ascending=[False, False]).head(10)['cmdb_id'].tolist()

# 2) Prepare resource-related metric rows in the incident window for selected components
# Ensure metric_window exists and has relevant rows
mw = metric_window.copy()  # metric_window created previously: rows within the incident window
# Join with resource_pairs to keep only resource-related KPI pairs that passed the earlier filter
rp_cols = ['cmdb_id','kpi_name','global_P95','global_P5']
mw_resource = mw.merge(resource_pairs[rp_cols].drop_duplicates(), on=['cmdb_id','kpi_name'], how='inner')

# Limit to top components
mw_resource = mw_resource[mw_resource['cmdb_id'].isin(top_components)].copy()
mw_resource.sort_values(['cmdb_id','kpi_name','timestamp'], inplace=True)

# 3) Mark anomaly points using global thresholds (pre-computed)
mw_resource['is_anomaly'] = (mw_resource['value'] >= mw_resource['global_P95']) | (mw_resource['value'] <= mw_resource['global_P5'])

# Focus on anomaly rows only to build consecutive runs
anoms = mw_resource[mw_resource['is_anomaly']].copy()

# If no anomaly rows, prepare empty outputs
if anoms.empty:
    all_runs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_points_count',
        'max_value_in_run','min_value_in_run','global_P95','global_P5','severity'
    ])
    result_top20 = all_runs.head(20)
    earliest_run_summary = None
else:
    # 3) & 4) Identify consecutive runs per (cmdb_id,kpi_name)
    def assign_runs(df):
        df = df.sort_values('timestamp').copy()
        df['gap'] = df['timestamp'].diff().fillna(60)  # in seconds; per-minute series expected
        # Start a new run when gap != 60
        df['run_segment'] = (df['gap'] != 60).cumsum()
        return df

    anoms = anoms.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_runs)

    # Aggregate runs
    run_agg = (
        anoms
        .groupby(['cmdb_id','kpi_name','run_segment'], as_index=False)
        .agg(
            run_start_ts=('timestamp','min'),
            run_end_ts=('timestamp','max'),
            anomaly_points_count=('timestamp','count'),
            max_value_in_run=('value','max'),
            min_value_in_run=('value','min'),
            global_P95=('global_P95','first'),
            global_P5=('global_P5','first')
        )
    )

    # Compute run_length_minutes (inclusive per-minute points -> length = count minutes)
    run_agg['run_length_minutes'] = run_agg['anomaly_points_count']  # each point is one minute

    # Compute severity per run using same rule, handle division carefully (inf when dividing by zero)
    def compute_run_severity(row):
        gp95 = row['global_P95']
        gp5 = row['global_P5']
        maxv = row['max_value_in_run']
        minv = row['min_value_in_run']
        # Prefer high-side severity
        if pd.notna(maxv) and pd.notna(gp95) and (maxv > gp95):
            if gp95 == 0:
                return float('inf')
            return float(maxv / gp95)
        if pd.notna(minv) and pd.notna(gp5) and (minv < gp5):
            if minv == 0:
                return float('inf')
            return float(gp5 / minv)
        return 1.0

    run_agg['severity'] = run_agg.apply(compute_run_severity, axis=1)

    # 4) Filter runs: run_length >= 3 and severity > 1.5
    filtered_runs = run_agg[(run_agg['run_length_minutes'] >= 3) & (run_agg['severity'] > 1.5)].copy()

    # Convert timestamps to ISO Asia/Shanghai for display
    def to_iso_opt(ts):
        return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

    filtered_runs['run_start_iso'] = filtered_runs['run_start_ts'].apply(to_iso_opt)
    filtered_runs['run_end_iso'] = filtered_runs['run_end_ts'].apply(to_iso_opt)

    # Prepare final rows with requested columns
    all_runs = filtered_runs.loc[:, [
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes','anomaly_points_count',
        'max_value_in_run','min_value_in_run','global_P95','global_P5','severity'
    ]].copy()

    # 6) Sort by run_start_ts ascending then severity descending, limit top 20
    result_top20 = all_runs.sort_values(by=['run_start_ts','severity'], ascending=[True, False]).head(20).reset_index(drop=True)

    # Also prepare earliest run among returned
    if not result_top20.empty:
        earliest = result_top20.sort_values(by='run_start_ts', ascending=True).iloc[0]
        earliest_run_summary = {
            'cmdb_id': earliest['cmdb_id'],
            'kpi_name': earliest['kpi_name'],
            'run_start_ts': int(earliest['run_start_ts']),
            'run_start_iso': to_iso_opt(int(earliest['run_start_ts'])),
            'severity': float(earliest['severity'])
        }
    else:
        earliest_run_summary = None

# Keep full runs in kernel variable 'all_runs' and filtered subset in 'result_top20'
all_runs, result_top20, earliest_run_summary

```
Out[4]:
```
Summary (plain English, compact):

- Scope: I inspected resource-related KPI series for the top 10 components by number of anomalous KPIs inside the incident window (2022-03-21 14:30:00 to 15:00:00 Asia/Shanghai), looking for consecutive anomalous runs of >=3 minutes and severity > 1.5.

- Matches found: 1 run passed the filters (returned as the top result, limited to 20).

Top runs (sorted by run_start then severity) — limited to top 20 (only 1 returned):
1) cmdb_id: frontend2-0
   - kpi_name: container.node-6.container_cpu_cfs_throttled_seconds
   - run_start_ts: 1647844500 (2022-03-21T14:35:00+08:00)
   - run_end_ts: 1647844620 (2022-03-21T14:37:00+08:00)
   - run_length_minutes: 3
   - anomaly_points_count: 3
   - max_value_in_run: 0.119682
   - min_value_in_run: 0.007573
   - global_P95: 0.119682
   - global_P5: 0.012725
   - severity: 1.680302

- Earliest run among the returned results:
  - component: frontend2-0
  - kpi: container.node-6.container_cpu_cfs_throttled_seconds
  - run_start: 1647844500 (2022-03-21T14:35:00+08:00)
  - severity: 1.680302

Interpretation:
- A 3-minute consecutive anomalous run was detected on frontend2-0 for CPU throttling (cfs throttled seconds) starting at 14:35 local time with severity ≈1.68. This indicates sustained throttling above the component's global P95 baseline and merits investigation (CPU throttling could point to container CPU limits or node resource pressure).

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes  anomaly_points_count  max_value_in_run  min_value_in_run  global_P95  global_P5  severity
2056  frontend2-0  container.node-6.container_cpu_cfs_throttled_s...    1647844500  1647844620                   3                     3          0.119682          0.007573    0.119682   0.012725  1.680302,        cmdb_id                                           kpi_name  run_start_ts  run_end_ts  run_length_minutes  anomaly_points_count  max_value_in_run  min_value_in_run  global_P95  global_P5  severity
0  frontend2-0  container.node-6.container_cpu_cfs_throttled_s...    1647844500  1647844620                   3                     3          0.119682          0.007573    0.119682   0.012725  1.680302, {'cmdb_id': 'frontend2-0', 'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'run_start_ts': 1647844500, 'run_start_iso': '2022-03-21T14:35:00+08:00', 'severity': 1.680301968123576})```
```

In [ ]:
# In[5]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df from kernel
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series (store as trace_thresholds)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(
        total_points_full='count',
        global_P95=lambda x: float(x.quantile(0.95)),
        global_P5=lambda x: float(x.quantile(0.05))
    )
)

# 2) Filter trace rows to the incident window
start_ts = 1647844200
end_ts = 1647846000
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Select rows meeting ANY of the conditions:
# cmdb_id == 'frontend2-0' OR trace_name contains 'from_frontend2-0' OR 'from_frontend-2' OR 'from_frontend' OR trace_name contains 'to_'
# Use case-insensitive matching on trace_name
cond_cmdb = trace_window['cmdb_id'] == 'frontend2-0'
tn_lower = trace_window['trace_name'].str.lower().fillna('')
cond_trace = (
    tn_lower.str.contains('from_frontend2-0') |
    tn_lower.str.contains('from_frontend-2') |
    tn_lower.str.contains('from_frontend') |
    tn_lower.str.contains('to_')
)
selected_traces = trace_window[cond_cmdb | cond_trace].copy()

# 4) For each selected (cmdb_id, trace_name) calculate anomaly_count etc using global thresholds (merge thresholds)
sel_with_thr = selected_traces.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Mark anomaly points in window
sel_with_thr['is_anomaly'] = (sel_with_thr['value'] >= sel_with_thr['global_P95']) | (sel_with_thr['value'] <= sel_with_thr['global_P5'])

# Aggregate per pair in window
agg_window = (
    sel_with_thr
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        total_points_in_window=('value','count'),
        anomaly_count=('is_anomaly','sum'),
        earliest_anomaly_ts=('timestamp', lambda s: int(s[sel_with_thr.loc[s.index,'is_anomaly']].min()) if any(sel_with_thr.loc[s.index,'is_anomaly']) else None),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    )
)

# Safer earliest_anomaly_ts calculation (handle groups without anomalies)
anom_rows = sel_with_thr[sel_with_thr['is_anomaly']]
if not anom_rows.empty:
    earliest_series = anom_rows.groupby(['cmdb_id','trace_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
    agg_window = agg_window.drop(columns=['earliest_anomaly_ts']).merge(earliest_series, on=['cmdb_id','trace_name'], how='left')
    agg_window['earliest_anomaly_ts'] = agg_window['earliest_anomaly_ts'].apply(lambda x: int(x) if pd.notna(x) else None)
else:
    agg_window['earliest_anomaly_ts'] = None

# 5) Prepare compact list sorted by anomaly_count desc then earliest_anomaly_ts asc, limit top 20
# Fill NaNs for sorting purposes
agg_window['anomaly_count'] = agg_window['anomaly_count'].astype(int)
agg_window_sorted = agg_window.sort_values(by=['anomaly_count','earliest_anomaly_ts'], ascending=[False, True])

agg_top20 = agg_window_sorted.loc[:, [
    'cmdb_id','trace_name','total_points_in_window','global_P95','global_P5',
    'anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window'
]].head(20).reset_index(drop=True)

# 6) Extract downstream target identifiers from anomalous trace rows (use anomalous rows in the window)
# Use regex to find tokens after 'to_' or 'from_'
anomalous_sel = sel_with_thr[sel_with_thr['is_anomaly']].copy()
def extract_targets(name):
    if pd.isna(name):
        return []
    name_l = name.lower()
    return re.findall(r'\b(?:to|from)_([a-z0-9\-]+)', name_l)

anomalous_sel['targets'] = anomalous_sel['trace_name'].apply(extract_targets)

# Explode and count
targets_exploded = anomalous_sel.explode('targets')
targets_exploded = targets_exploded[targets_exploded['targets'].notna() & (targets_exploded['targets'] != '')]

if not targets_exploded.empty:
    downstream_counts = (
        targets_exploded
        .groupby('targets', as_index=False)
        .agg(anomaly_occurrences=('trace_name','count'))
        .sort_values(by='anomaly_occurrences', ascending=False)
    )
    downstream_counts_top20 = downstream_counts.head(20).rename(columns={'targets':'service_identifier'})
else:
    downstream_counts_top20 = pd.DataFrame(columns=['service_identifier','anomaly_occurrences'])

# Store full intermediate tables in kernel variables (trace_thresholds, trace_window, selected_traces, sel_with_thr, agg_window, anomalous_sel, targets_exploded)
trace_thresholds, trace_window, selected_traces, sel_with_thr, agg_window, anomalous_sel, targets_exploded, agg_top20, downstream_counts_top20

```
Out[5]:
```
Summary (plain English, compact):

- Scope & preprocessing:
  - Computed global thresholds (P95/P5) for 1,232 unique (cmdb_id, trace_name) pairs from trace.csv (stored in kernel).
  - Incident-window rows (timestamps 1647844200–1647846000): 36,312 total trace samples.
  - After applying selection criteria (cmdb_id == 'frontend2-0' OR trace_name contains from_frontend*/to_*), 25,256 trace rows remained.
  - These map to 856 distinct (cmdb_id, trace_name) pairs in the window (aggregated results stored in kernel).

- Anomaly summary (per-pair aggregation, top behavior):
  - The aggregation produced 856 pairs with per-window stats (total_points_in_window, global_P95, global_P5, anomaly_count, earliest_anomaly_ts, max/min in window).
  - Many trace pairs show high anomaly_count (often all 30 points in-window flagged). Example top rows:
    - adservice-0 · trace.from_frontend-0.error_rate — total_points_in_window=30, global_P95=0.0, global_P5=0.0, anomaly_count=30, earliest_anomaly_ts=1647844200, max/min=0.0
    - adservice-0 · trace.from_frontend-0.duration_mean — 30 points, anomaly_count=6, earliest_anomaly_ts=1647844380, max≈0.000026, min≈0.000015
    - adservice-0 · trace.from_frontend-0.row_count — 30 points, anomaly_count=4, earliest_anomaly_ts≈1647844740, max=18, min=12
  - Many error_rate trace metrics are exactly zero across the window and are flagged as anomalies due to global thresholds being zero — earliest anomalies often at the window start (1647844200).

- Downstream targets (from anomalous trace rows):
  - Extracting 'to_' / 'from_' identifiers from anomalous rows yields these top downstream identifiers by anomaly occurrences (top 20 shown; counts are anomaly-point occurrences):
    - frontend-0: 1102
    - frontend-2: 1077
    - frontend-1: 1055
    - cartservice-2: 513
    - productcatalogservice-2: 498
    - productcatalogservice-1: 491
    - productcatalogservice-0: 491
    - cartservice-0: 486
    - cartservice-1: 485
    - frontend2-0: 339
    - shippingservice-0: 304
    - shippingservice-1: 289
    - shippingservice-2: 285
    - currencyservice-1: 284
    - currencyservice-2: 280
    - currencyservice-0: 270
    - paymentservice-2: 196
    - emailservice-1: 192
    - emailservice-0: 192
    - paymentservice-0: 191

- Interpretation & caveats:
  - Many anomalous trace signals point to frontend-originated paths and downstream services (row counts, durations, and error_rate).
  - A large fraction of anomalies are driven by zero-valued error_rate metrics where global thresholds are zero — these produce many anomaly flags at the window start and may be artifacts of thresholding rather than meaningful errors.
  - Non-zero anomalies (e.g., elevated row_count or duration) appear for several services (adservice, shippingservice, productcatalogservice, cartservice, etc.) and should be correlated with metric anomalies (CPU/IO) previously found.

- Recommended next actions:
  - Investigate frontends and the top downstream services listed above (start with frontend-0/1/2, cartservice*, productcatalogservice*, shippingservice*).
  - Inspect the raw trace time series for row_count and duration metrics (not just error_rate zeros) and correlate with metric-level resource anomalies (CPU throttling, IO, memory failures) already identified.
  - Revisit global threshold treatment for zero-valued trace KPIs (avoid treating zeros as hard anomalies without context).

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                               trace_name  total_points_full  global_P95  global_P5
0            adservice-0      trace.from_frontend-0.duration_mean                 30    0.000026   0.000016
1            adservice-0       trace.from_frontend-0.duration_p95                 30    0.000059   0.000018
2            adservice-0         trace.from_frontend-0.error_rate                 30    0.000000   0.000000
3            adservice-0          trace.from_frontend-0.row_count                 30   17.550000  12.450000
4            adservice-0      trace.from_frontend-1.duration_mean                 30    0.000025   0.000016
...                  ...                                      ...                ...         ...        ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count                 30   14.000000   6.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean                 30    0.000083   0.000070
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95                 30    0.000093   0.000077
1230  shippingservice2-0        trace.from_frontend2-0.error_rate                 30    0.000000   0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count                 30   32.000000  18.450000

[1232 rows x 5 columns],         timestamp             cmdb_id                               trace_name      value
0      1647844200         adservice-0      trace.from_frontend-0.duration_mean   0.000017
1      1647844200         adservice-0       trace.from_frontend-0.duration_p95   0.000023
2      1647844200         adservice-0         trace.from_frontend-0.error_rate   0.000000
3      1647844200         adservice-0          trace.from_frontend-0.row_count  17.000000
4      1647844200         adservice-0      trace.from_frontend-1.duration_mean   0.000017
...           ...                 ...                                      ...        ...
36307  1647845940  shippingservice2-0  trace.from_checkoutservice2-0.row_count   8.000000
36308  1647845940  shippingservice2-0     trace.from_frontend2-0.duration_mean   0.000073
36309  1647845940  shippingservice2-0      trace.from_frontend2-0.duration_p95   0.000085
36310  1647845940  shippingservice2-0        trace.from_frontend2-0.error_rate   0.000000
36311  1647845940  shippingservice2-0         trace.from_frontend2-0.row_count  18.000000

[36312 rows x 4 columns],         timestamp             cmdb_id                            trace_name      value
0      1647844200         adservice-0   trace.from_frontend-0.duration_mean   0.000017
1      1647844200         adservice-0    trace.from_frontend-0.duration_p95   0.000023
2      1647844200         adservice-0      trace.from_frontend-0.error_rate   0.000000
3      1647844200         adservice-0       trace.from_frontend-0.row_count  17.000000
4      1647844200         adservice-0   trace.from_frontend-1.duration_mean   0.000017
...           ...                 ...                                   ...        ...
36303  1647845940   shippingservice-2       trace.from_frontend-2.row_count   7.000000
36308  1647845940  shippingservice2-0  trace.from_frontend2-0.duration_mean   0.000073
36309  1647845940  shippingservice2-0   trace.from_frontend2-0.duration_p95   0.000085
36310  1647845940  shippingservice2-0     trace.from_frontend2-0.error_rate   0.000000
36311  1647845940  shippingservice2-0      trace.from_frontend2-0.row_count  18.000000

[25256 rows x 4 columns],         timestamp             cmdb_id                            trace_name      value  total_points_full  global_P95  global_P5  is_anomaly
0      1647844200         adservice-0   trace.from_frontend-0.duration_mean   0.000017                 30    0.000026   0.000016       False
1      1647844200         adservice-0    trace.from_frontend-0.duration_p95   0.000023                 30    0.000059   0.000018       False
2      1647844200         adservice-0      trace.from_frontend-0.error_rate   0.000000                 30    0.000000   0.000000        True
3      1647844200         adservice-0       trace.from_frontend-0.row_count  17.000000                 30   17.550000  12.450000       False
4      1647844200         adservice-0   trace.from_frontend-1.duration_mean   0.000017                 30    0.000025   0.000016       False
...           ...                 ...                                   ...        ...                ...         ...        ...         ...
25251  1647845940   shippingservice-2       trace.from_frontend-2.row_count   7.000000                 30    9.000000   3.000000       False
25252  1647845940  shippingservice2-0  trace.from_frontend2-0.duration_mean   0.000073                 30    0.000083   0.000070       False
25253  1647845940  shippingservice2-0   trace.from_frontend2-0.duration_p95   0.000085                 30    0.000093   0.000077       False
25254  1647845940  shippingservice2-0     trace.from_frontend2-0.error_rate   0.000000                 30    0.000000   0.000000        True
25255  1647845940  shippingservice2-0      trace.from_frontend2-0.row_count  18.000000                 30   32.000000  18.450000        True

[25256 rows x 8 columns],                 cmdb_id                            trace_name  total_points_in_window  anomaly_count  max_value_in_window  min_value_in_window  global_P95  global_P5  earliest_anomaly_ts
0           adservice-0   trace.from_frontend-0.duration_mean                      30              6             0.000026             0.000015    0.000026   0.000016           1647844380
1           adservice-0    trace.from_frontend-0.duration_p95                      30              4             0.000060             0.000017    0.000059   0.000018           1647844440
2           adservice-0      trace.from_frontend-0.error_rate                      30             30             0.000000             0.000000    0.000000   0.000000           1647844200
3           adservice-0       trace.from_frontend-0.row_count                      30              4            18.000000            12.000000   17.550000  12.450000           1647844740
4           adservice-0   trace.from_frontend-1.duration_mean                      30              5             0.000025             0.000016    0.000025   0.000016           1647844440
..                  ...                                   ...                     ...            ...                  ...                  ...         ...        ...                  ...
851   shippingservice-2       trace.from_frontend-2.row_count                      30              8             9.000000             3.000000    9.000000   3.000000           1647845160
852  shippingservice2-0  trace.from_frontend2-0.duration_mean                      30              6             0.000085             0.000068    0.000083   0.000070           1647844380
853  shippingservice2-0   trace.from_frontend2-0.duration_p95                      30              4             0.000109             0.000073    0.000093   0.000077           1647844440
854  shippingservice2-0     trace.from_frontend2-0.error_rate                      30             30             0.000000             0.000000    0.000000   0.000000           1647844200
855  shippingservice2-0      trace.from_frontend2-0.row_count                      30              5            33.000000            14.000000   32.000000  18.450000           1647845640

[856 rows x 9 columns],         timestamp             cmdb_id                         trace_name  value  total_points_full  global_P95  global_P5  is_anomaly        targets
2      1647844200         adservice-0   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   [frontend-0]
6      1647844200         adservice-0   trace.from_frontend-1.error_rate    0.0                 30         0.0       0.00        True   [frontend-1]
10     1647844200         adservice-0   trace.from_frontend-2.error_rate    0.0                 30         0.0       0.00        True   [frontend-2]
14     1647844200         adservice-1   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   [frontend-0]
15     1647844200         adservice-1    trace.from_frontend-0.row_count   17.0                 30        17.0      13.00        True   [frontend-0]
...           ...                 ...                                ...    ...                ...         ...        ...         ...            ...
25242  1647845940   shippingservice-2   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   [frontend-0]
25246  1647845940   shippingservice-2   trace.from_frontend-1.error_rate    0.0                 30         0.0       0.00        True   [frontend-1]
25250  1647845940   shippingservice-2   trace.from_frontend-2.error_rate    0.0                 30         0.0       0.00        True   [frontend-2]
25254  1647845940  shippingservice2-0  trace.from_frontend2-0.error_rate    0.0                 30         0.0       0.00        True  [frontend2-0]
25255  1647845940  shippingservice2-0   trace.from_frontend2-0.row_count   18.0                 30        32.0      18.45        True  [frontend2-0]

[11587 rows x 9 columns],         timestamp             cmdb_id                         trace_name  value  total_points_full  global_P95  global_P5  is_anomaly      targets
2      1647844200         adservice-0   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   frontend-0
6      1647844200         adservice-0   trace.from_frontend-1.error_rate    0.0                 30         0.0       0.00        True   frontend-1
10     1647844200         adservice-0   trace.from_frontend-2.error_rate    0.0                 30         0.0       0.00        True   frontend-2
14     1647844200         adservice-1   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   frontend-0
15     1647844200         adservice-1    trace.from_frontend-0.row_count   17.0                 30        17.0      13.00        True   frontend-0
...           ...                 ...                                ...    ...                ...         ...        ...         ...          ...
25242  1647845940   shippingservice-2   trace.from_frontend-0.error_rate    0.0                 30         0.0       0.00        True   frontend-0
25246  1647845940   shippingservice-2   trace.from_frontend-1.error_rate    0.0                 30         0.0       0.00        True   frontend-1
25250  1647845940   shippingservice-2   trace.from_frontend-2.error_rate    0.0                 30         0.0       0.00        True   frontend-2
25254  1647845940  shippingservice2-0  trace.from_frontend2-0.error_rate    0.0                 30         0.0       0.00        True  frontend2-0
25255  1647845940  shippingservice2-0   trace.from_frontend2-0.row_count   18.0                 30        32.0      18.45        True  frontend2-0

[11545 rows x 9 columns],            cmdb_id                           trace_name  total_points_in_window  global_P95  global_P5  anomaly_count  earliest_anomaly_ts  max_value_in_window  min_value_in_window
0      adservice-0     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
1      adservice-0     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
2      adservice-0     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
3      adservice-1     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
4      adservice-1     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
5      adservice-1     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
6      adservice-2     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
7      adservice-2     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
8      adservice-2     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
9     adservice2-0    trace.from_frontend2-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
10   cartservice-0     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
11   cartservice-0     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
12   cartservice-0     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
13   cartservice-1     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
14   cartservice-1     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
15   cartservice-1     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
16   cartservice-2     trace.from_frontend-0.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
17   cartservice-2     trace.from_frontend-1.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
18   cartservice-2     trace.from_frontend-2.error_rate                      30       0.000      0.000             30           1647844200                0.000                0.000
19  cartservice2-0  trace.from_frontend2-0.duration_p95                      30       0.001      0.001             30           1647844200                0.001                0.001,          service_identifier  anomaly_occurrences
20               frontend-0                 1102
22               frontend-2                 1077
21               frontend-1                 1055
6             cartservice-2                  513
30  productcatalogservice-2                  498
29  productcatalogservice-1                  491
28  productcatalogservice-0                  491
4             cartservice-0                  486
5             cartservice-1                  485
23              frontend2-0                  339
37        shippingservice-0                  304
38        shippingservice-1                  289
39        shippingservice-2                  285
13        currencyservice-1                  284
14        currencyservice-2                  280
12        currencyservice-0                  270
26         paymentservice-2                  196
17           emailservice-1                  192
16           emailservice-0                  192
24         paymentservice-0                  191)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing DataFrames from kernel: log_df, error_logs_df
# Ensure correct dtypes
log_df['timestamp'] = log_df['timestamp'].astype(int)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
error_logs_df['timestamp'] = error_logs_df['timestamp'].astype(int)

# Parameters
start_ts = 1647844200
end_ts = 1647846000
selected_cmdbs = ['frontend2-0','frontend-0','frontend-1','frontend-2','adservice-0','adservice','checkoutservice-0','node-5']

# 1) Global thresholds per (cmdb_id, log_name) using full series (store in kernel)
log_thresholds = (
    log_df
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(
        total_points_full='count',
        global_P95=lambda x: float(x.quantile(0.95)),
        global_P5=lambda x: float(x.quantile(0.05))
    )
)

# 2) Filter log rows to incident window
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) Restrict to selected cmdb_ids
sel_logs = log_window[log_window['cmdb_id'].isin(selected_cmdbs)].copy()

# 4) Merge thresholds and mark anomalies
sel_logs = sel_logs.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
sel_logs['is_anomaly'] = (sel_logs['value'] >= sel_logs['global_P95']) | (sel_logs['value'] <= sel_logs['global_P5'])

# Aggregate per (cmdb_id, log_name)
# earliest_anomaly_ts computed safely after grouping anomalies
agg_basic = (
    sel_logs
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(
        total_points_in_window=('value','count'),
        anomaly_count=('is_anomaly','sum'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    )
)

# earliest anomaly ts
anom_rows = sel_logs[sel_logs['is_anomaly']].copy()
if not anom_rows.empty:
    earliest = anom_rows.groupby(['cmdb_id','log_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
    agg_basic = agg_basic.merge(earliest, on=['cmdb_id','log_name'], how='left')
else:
    agg_basic['earliest_anomaly_ts'] = None

# Add ISO representation for compact readability
def to_iso_opt(ts):
    if pd.isna(ts) or ts is None:
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

agg_basic['earliest_anomaly_iso'] = agg_basic['earliest_anomaly_ts'].apply(to_iso_opt)

# 5) Locate consecutive anomalous runs (per pair) and keep runs with run_length >=3 minutes
anom_only = sel_logs[sel_logs['is_anomaly']].copy()
runs_df = pd.DataFrame(columns=[
    'cmdb_id','log_name','run_start_ts','run_end_ts','run_length_minutes',
    'anomaly_points_count','max_value_in_run','min_value_in_run'
])

if not anom_only.empty:
    def assign_run_segments(df):
        df = df.sort_values('timestamp').copy()
        # difference in seconds
        df['gap'] = df['timestamp'].diff().fillna(60)
        df['run_seg'] = (df['gap'] != 60).cumsum()
        return df

    anom_seg = anom_only.groupby(['cmdb_id','log_name'], group_keys=False).apply(assign_run_segments)

    run_agg = (
        anom_seg
        .groupby(['cmdb_id','log_name','run_seg'], as_index=False)
        .agg(
            run_start_ts=('timestamp','min'),
            run_end_ts=('timestamp','max'),
            anomaly_points_count=('timestamp','count'),
            max_value_in_run=('value','max'),
            min_value_in_run=('value','min')
        )
    )
    run_agg['run_length_minutes'] = run_agg['anomaly_points_count']  # one point per minute

    # Keep runs with length >=3
    runs_df = run_agg[run_agg['run_length_minutes'] >= 3].copy()
    # reorder columns
    runs_df = runs_df.loc[:, ['cmdb_id','log_name','run_start_ts','run_end_ts','run_length_minutes',
                              'anomaly_points_count','max_value_in_run','min_value_in_run']]

    # Add ISO columns for readability
    runs_df['run_start_iso'] = runs_df['run_start_ts'].apply(to_iso_opt)
    runs_df['run_end_iso'] = runs_df['run_end_ts'].apply(to_iso_opt)

# 6A) Per-(cmdb_id,log_name) summary sorted and limited to top 20
per_pair_summary = agg_basic.copy()
per_pair_summary = per_pair_summary.sort_values(by=['anomaly_count','earliest_anomaly_ts'], ascending=[False, True])
per_pair_summary_top20 = per_pair_summary.loc[:, [
    'cmdb_id','log_name','total_points_in_window','global_P95','global_P5',
    'anomaly_count','earliest_anomaly_ts','earliest_anomaly_iso','max_value_in_window','min_value_in_window'
]].head(20).reset_index(drop=True)

# 6B) Consecutive-run table sorted by run_start_ts asc then run_length desc, limited to top 20
runs_top20 = pd.DataFrame()
if not runs_df.empty:
    runs_top20 = runs_df.sort_values(by=['run_start_ts','run_length_minutes'], ascending=[True, False]).head(20).reset_index(drop=True)

# 7) Search error_logs.csv for matching cmdb_ids and timestamps in window
err_matches = error_logs_df[
    (error_logs_df['timestamp'] >= start_ts) &
    (error_logs_df['timestamp'] <= end_ts) &
    (error_logs_df['cmdb_id'].isin(selected_cmdbs))
].copy()

if err_matches.empty:
    error_logs_matches = "No error_logs entries found for selected components in the window."
else:
    # compact selection of columns and limit to 20
    error_logs_matches = err_matches.loc[:, ['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# Keep full intermediate tables in kernel variables (log_thresholds, log_window, sel_logs, agg_basic, runs_df, anom_only, err_matches)
log_thresholds, log_window, sel_logs, agg_basic, runs_df, per_pair_summary_top20, runs_top20, error_logs_matches

```
Out[6]:
```
Summary (plain English, compact):

- What I checked:
  - Computed global P95/P5 for each (cmdb_id, log_name) using the full log series, and examined logs inside the incident window 2022-03-21 14:30:00–15:00:00 (Asia/Shanghai) for these components: frontend2-0, frontend-0, frontend-1, frontend-2, adservice-0, adservice, checkoutservice-0, node-5.
  - Also searched error_logs.csv for matching entries in the same window.

- Key findings (logs):
  - Many services show log.error_count == 0 across the window and therefore appear as “anomalous” by threshold rules (global P95/P5 == 0). Those zero-error flags are widespread for frontends and others and are likely thresholding artifacts, not direct evidence of a failure.
  - Frontend log row_count anomalies:
    - frontend-1: row_count had anomaly_count=4, earliest anomaly ~2022-03-21T14:32:00+08, max 1598.
    - frontend-0: row_count anomaly_count=4, earliest ~2022-03-21T14:37:00+08, max 1338.
    - frontend-2: row_count anomaly_count=4, earliest ~2022-03-21T14:41:00+08, max 1571.
    - adservice-0 and checkoutservice-0 also show row_count anomalous runs (3-minute runs observed).
  - Consecutive anomalous runs of length >= 3 minutes were found for several log metrics (e.g., persistent log.error_count zeros across full window for frontends and adservice/checkoutservice; 3-min row_count runs for adservice-0 and checkoutservice-0). 
  - Crucially: frontend2-0 does not appear in the per-(cmdb_id,log_name) top summary and no 3+ minute anomalous log runs for frontend2-0 were returned — i.e., there is no clear log.signal (log.row_count or log.error_count) inside the window that corroborates the frontend2-0 CPU throttling event.

- Error logs:
  - No error_logs entries found for the selected components in the incident window.

- Interpretation and conclusion:
  - The log and error_log data do not provide supporting evidence that frontend2-0’s CPU throttling (observed earlier in the metric analysis) produced application-level errors or increased error_count/row_count in its own logs during the window.
  - Other frontends (frontend-0/1/2) and services (adservice-0, checkoutservice-0) show log anomalies (row_count spikes and long zero error_count runs), but these do not specifically implicate frontend2-0 as the source based on logs alone.
  - Many anomaly flags are driven by zero-valued error_count baselines — treat those with caution.

- Recommended next steps:
  - Investigate the metric-based CPU throttling signal on frontend2-0 further (container CPU limits, cgroup settings, node CPU pressure) since logs do not confirm an application-level impact.
  - Correlate the frontend2-0 CPU throttling time window with traces (requests from frontend2-0) and node-level metrics (node-5) to see if throttling aligns with increased latency, downstream row_count/duration changes, or node resource contention.
  - Inspect host/container runtime logs (not in metric/log CSVs) for CPU throttling messages and check orchestration events (e.g., OOM, CPU throttling events) on the node hosting frontend2-0.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id         log_name  total_points_full  global_P95  global_P5
0         adservice-0  log.error_count                 30         0.0        0.0
1         adservice-0    log.row_count                 30       127.1      103.8
2         adservice-1  log.error_count                 30         0.0        0.0
3         adservice-1    log.row_count                 30       114.2       92.0
4         adservice-2  log.error_count                 30         0.0        0.0
..                ...              ...                ...         ...        ...
57  shippingservice-0    log.row_count                 31       110.0       61.5
58  shippingservice-1  log.error_count                 30         0.0        0.0
59  shippingservice-1    log.row_count                 30       113.4       63.5
60  shippingservice-2  log.error_count                 31         0.0        0.0
61  shippingservice-2    log.row_count                 31       109.0       64.5

[62 rows x 5 columns],        timestamp                  cmdb_id         log_name  value
0     1647844200              adservice-0  log.error_count    0.0
1     1647844200              adservice-0    log.row_count  122.0
2     1647844200              adservice-1  log.error_count    0.0
3     1647844200              adservice-1    log.row_count  112.0
4     1647844200              adservice-2  log.error_count    0.0
...          ...                      ...              ...    ...
1837  1647846000  recommendationservice-1    log.row_count    2.0
1838  1647846000        shippingservice-0  log.error_count    0.0
1839  1647846000        shippingservice-0    log.row_count    2.0
1840  1647846000        shippingservice-2  log.error_count    0.0
1841  1647846000        shippingservice-2    log.row_count    2.0

[1842 rows x 4 columns],       timestamp            cmdb_id         log_name  value  total_points_full  global_P95  global_P5  is_anomaly
0    1647844200        adservice-0  log.error_count    0.0                 30        0.00        0.0        True
1    1647844200        adservice-0    log.row_count  122.0                 30      127.10      103.8       False
2    1647844200  checkoutservice-0  log.error_count    0.0                 30        0.00        0.0        True
3    1647844200  checkoutservice-0    log.row_count   45.0                 30       75.95       34.0       False
4    1647844200         frontend-0  log.error_count    0.0                 31        0.00        0.0        True
..          ...                ...              ...    ...                ...         ...        ...         ...
301  1647846000         frontend-0    log.row_count    9.0                 31     1295.00     1020.0        True
302  1647846000         frontend-1  log.error_count    0.0                 31        0.00        0.0        True
303  1647846000         frontend-1    log.row_count    2.0                 31     1578.00     1218.5        True
304  1647846000         frontend-2  log.error_count    0.0                 31        0.00        0.0        True
305  1647846000         frontend-2    log.row_count    3.0                 31     1509.00     1207.5        True

[306 rows x 8 columns],              cmdb_id         log_name  total_points_in_window  anomaly_count  max_value_in_window  min_value_in_window  global_P95  global_P5  earliest_anomaly_ts       earliest_anomaly_iso
0        adservice-0  log.error_count                      30             30                  0.0                  0.0        0.00        0.0           1647844200  2022-03-21T14:30:00+08:00
1        adservice-0    log.row_count                      30              4                128.0                102.0      127.10      103.8           1647844560  2022-03-21T14:36:00+08:00
2  checkoutservice-0  log.error_count                      30             30                  0.0                  0.0        0.00        0.0           1647844200  2022-03-21T14:30:00+08:00
3  checkoutservice-0    log.row_count                      30              6                 81.0                 33.0       75.95       34.0           1647844920  2022-03-21T14:42:00+08:00
4         frontend-0  log.error_count                      31             31                  0.0                  0.0        0.00        0.0           1647844200  2022-03-21T14:30:00+08:00
5         frontend-0    log.row_count                      31              4               1338.0                  9.0     1295.00     1020.0           1647844620  2022-03-21T14:37:00+08:00
6         frontend-1  log.error_count                      31             31                  0.0                  0.0        0.00        0.0           1647844200  2022-03-21T14:30:00+08:00
7         frontend-1    log.row_count                      31              4               1598.0                  2.0     1578.00     1218.5           1647844320  2022-03-21T14:32:00+08:00
8         frontend-2  log.error_count                      31             31                  0.0                  0.0        0.00        0.0           1647844200  2022-03-21T14:30:00+08:00
9         frontend-2    log.row_count                      31              4               1571.0                  3.0     1509.00     1207.5           1647844860  2022-03-21T14:41:00+08:00,               cmdb_id         log_name  run_start_ts  run_end_ts  run_length_minutes  anomaly_points_count  max_value_in_run  min_value_in_run              run_start_iso                run_end_iso
0         adservice-0  log.error_count    1647844200  1647845940                  30                    30               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
2         adservice-0    log.row_count    1647845460  1647845580                   3                     3             128.0             102.0  2022-03-21T14:51:00+08:00  2022-03-21T14:53:00+08:00
3   checkoutservice-0  log.error_count    1647844200  1647845940                  30                    30               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
4   checkoutservice-0    log.row_count    1647844920  1647845040                   3                     3              81.0              34.0  2022-03-21T14:42:00+08:00  2022-03-21T14:44:00+08:00
5   checkoutservice-0    log.row_count    1647845820  1647845940                   3                     3              80.0              33.0  2022-03-21T14:57:00+08:00  2022-03-21T14:59:00+08:00
6          frontend-0  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00
11         frontend-1  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00
16         frontend-2  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00,              cmdb_id         log_name  total_points_in_window  global_P95  global_P5  anomaly_count  earliest_anomaly_ts       earliest_anomaly_iso  max_value_in_window  min_value_in_window
0         frontend-0  log.error_count                      31        0.00        0.0             31           1647844200  2022-03-21T14:30:00+08:00                  0.0                  0.0
1         frontend-1  log.error_count                      31        0.00        0.0             31           1647844200  2022-03-21T14:30:00+08:00                  0.0                  0.0
2         frontend-2  log.error_count                      31        0.00        0.0             31           1647844200  2022-03-21T14:30:00+08:00                  0.0                  0.0
3        adservice-0  log.error_count                      30        0.00        0.0             30           1647844200  2022-03-21T14:30:00+08:00                  0.0                  0.0
4  checkoutservice-0  log.error_count                      30        0.00        0.0             30           1647844200  2022-03-21T14:30:00+08:00                  0.0                  0.0
5  checkoutservice-0    log.row_count                      30       75.95       34.0              6           1647844920  2022-03-21T14:42:00+08:00                 81.0                 33.0
6         frontend-1    log.row_count                      31     1578.00     1218.5              4           1647844320  2022-03-21T14:32:00+08:00               1598.0                  2.0
7        adservice-0    log.row_count                      30      127.10      103.8              4           1647844560  2022-03-21T14:36:00+08:00                128.0                102.0
8         frontend-0    log.row_count                      31     1295.00     1020.0              4           1647844620  2022-03-21T14:37:00+08:00               1338.0                  9.0
9         frontend-2    log.row_count                      31     1509.00     1207.5              4           1647844860  2022-03-21T14:41:00+08:00               1571.0                  3.0,              cmdb_id         log_name  run_start_ts  run_end_ts  run_length_minutes  anomaly_points_count  max_value_in_run  min_value_in_run              run_start_iso                run_end_iso
0         frontend-0  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00
1         frontend-1  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00
2         frontend-2  log.error_count    1647844200  1647846000                  31                    31               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T15:00:00+08:00
3        adservice-0  log.error_count    1647844200  1647845940                  30                    30               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
4  checkoutservice-0  log.error_count    1647844200  1647845940                  30                    30               0.0               0.0  2022-03-21T14:30:00+08:00  2022-03-21T14:59:00+08:00
5  checkoutservice-0    log.row_count    1647844920  1647845040                   3                     3              81.0              34.0  2022-03-21T14:42:00+08:00  2022-03-21T14:44:00+08:00
6        adservice-0    log.row_count    1647845460  1647845580                   3                     3             128.0             102.0  2022-03-21T14:51:00+08:00  2022-03-21T14:53:00+08:00
7  checkoutservice-0    log.row_count    1647845820  1647845940                   3                     3              80.0              33.0  2022-03-21T14:57:00+08:00  2022-03-21T14:59:00+08:00, 'No error_logs entries found for selected components in the window.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')
start_ts = 1647844200
end_ts = 1647846000

# Ensure trace_df and metric_df exist in kernel (they were loaded earlier)
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# ---- PART 1: Traces for frontend2-0 ----
# Compute global P95 and P5 per (cmdb_id, trace_name) using robust steps (avoid pandas .agg kwargs tuple issue)
grp = trace_df.groupby(['cmdb_id','trace_name'])['value']
trace_counts = grp.count().reset_index(name='total_points_full')
trace_p95 = grp.quantile(0.95).reset_index(name='global_P95')
trace_p5 = grp.quantile(0.05).reset_index(name='global_P5')
trace_thresholds = trace_counts.merge(trace_p95, on=['cmdb_id','trace_name']).merge(trace_p5, on=['cmdb_id','trace_name'])

# Filter trace rows for frontend2-0 within window
trace_frontend_window = trace_df[
    (trace_df['cmdb_id'] == 'frontend2-0') &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Merge thresholds and mark anomalies
trace_frontend_window = trace_frontend_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_frontend_window['is_anomaly'] = (trace_frontend_window['value'] >= trace_frontend_window['global_P95']) | \
                                      (trace_frontend_window['value'] <= trace_frontend_window['global_P5'])

# Aggregate per trace_name
agg_trace_frontend = (
    trace_frontend_window
    .groupby('trace_name', as_index=False)
    .agg(
        total_points_in_window=('value','count'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first'),
        anomaly_count=('is_anomaly','sum'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min')
    )
)

# earliest anomaly ts
anoms_trace = trace_frontend_window[trace_frontend_window['is_anomaly']].copy()
if not anoms_trace.empty:
    earliest_trace = anoms_trace.groupby('trace_name', as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
    agg_trace_frontend = agg_trace_frontend.merge(earliest_trace, on='trace_name', how='left')
else:
    agg_trace_frontend['earliest_anomaly_ts'] = None

# ISO column
agg_trace_frontend['earliest_anomaly_iso'] = agg_trace_frontend['earliest_anomaly_ts'].apply(
    lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(x) else None
)

# Sort and limit
trace_top20 = agg_trace_frontend.sort_values(by=['anomaly_count','earliest_anomaly_ts'], ascending=[False, True]) \
                               .loc[:, ['trace_name','total_points_in_window','global_P95','global_P5',
                                        'anomaly_count','earliest_anomaly_ts','earliest_anomaly_iso',
                                        'max_value_in_window','min_value_in_window']].head(20).reset_index(drop=True)

# One-line summary for trace.self.* metrics
mask_self = trace_top20['trace_name'].str.contains('trace.self', na=False)
self_rows = trace_top20[mask_self]
if not self_rows.empty and (self_rows['anomaly_count'].sum() > 0):
    earliest_self_ts = self_rows['earliest_anomaly_ts'].dropna().min()
    earliest_self_iso = pd.to_datetime(int(earliest_self_ts), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(earliest_self_ts) else None
    trace_self_summary = f"trace.self.* metrics for frontend2-0 have anomalies; earliest at {earliest_self_iso}"
else:
    trace_self_summary = "No trace.self.* metrics for frontend2-0 have anomaly_count>0 in the window."

# ---- PART 2: Node-level metrics for node-6 and node-5 ----
# Compute global thresholds for metric_df robustly
mgrp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
metric_counts = mgrp.count().reset_index(name='total_points_full')
metric_p95 = mgrp.quantile(0.95).reset_index(name='global_P95')
metric_p5 = mgrp.quantile(0.05).reset_index(name='global_P5')
metric_thresholds = metric_counts.merge(metric_p95, on=['cmdb_id','kpi_name']).merge(metric_p5, on=['cmdb_id','kpi_name'])

# Filter metric rows for node-6/node-5 in window and matching substrings
nodes = ['node-6','node-5']
substrs = ['cpu','iowait','io','util','disk','rkb','wkb','system.io','load']
node_window = metric_df[
    (metric_df['cmdb_id'].isin(nodes)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()
node_window['kpi_name_lc'] = node_window['kpi_name'].str.lower().fillna('')
node_window = node_window[node_window['kpi_name_lc'].apply(lambda s: any(sub in s for sub in substrs))].copy()

# Merge thresholds and mark anomalies
node_window = node_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
node_window['is_anomaly'] = (node_window['value'] >= node_window['global_P95']) | (node_window['value'] <= node_window['global_P5'])

# Aggregate per (cmdb_id,kpi_name)
agg_node = (
    node_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        total_points_in_window=('value','count'),
        anomaly_count=('is_anomaly','sum'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_P95=('global_P95','first'),
        global_P5=('global_P5','first')
    )
)

# earliest anomaly ts
anoms_node = node_window[node_window['is_anomaly']].copy()
if not anoms_node.empty:
    earliest_node = anoms_node.groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_ts'})
    agg_node = agg_node.merge(earliest_node, on=['cmdb_id','kpi_name'], how='left')
else:
    agg_node['earliest_anomaly_ts'] = None

# ISO column
agg_node['earliest_anomaly_iso'] = agg_node['earliest_anomaly_ts'].apply(
    lambda x: pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).isoformat() if pd.notna(x) else None
)

# Compute severity per spec, handle zero thresholds
def compute_sev(row):
    gp95 = row['global_P95']
    gp5 = row['global_P5']
    maxv = row['max_value_in_window']
    minv = row['min_value_in_window']
    if pd.notna(maxv) and pd.notna(gp95) and (maxv > gp95):
        if gp95 == 0:
            return float('inf')
        return float(maxv / gp95)
    if pd.notna(minv) and pd.notna(gp5) and (minv < gp5):
        if minv == 0:
            return float('inf')
        return float(gp5 / minv)
    return 1.0

agg_node['severity'] = agg_node.apply(compute_sev, axis=1)

# Sort by severity desc then anomaly_count desc, limit 20
node_top20 = agg_node.sort_values(by=['severity','anomaly_count'], ascending=[False, False]) \
                     .loc[:, ['cmdb_id','kpi_name','total_points_in_window','global_P95','global_P5',
                              'anomaly_count','earliest_anomaly_ts','earliest_anomaly_iso',
                              'max_value_in_window','min_value_in_window','severity']].head(20).reset_index(drop=True)

# One-line summary: which node shows largest severity and earliest anomaly ts
if not agg_node.empty:
    # consider inf as largest; handle by comparing after replacing inf with large number for argmax but keep original for message
    sev_for_argmax = agg_node['severity'].replace([np.inf], 1e308)
    idx_max = sev_for_argmax.idxmax()
    row_max = agg_node.loc[idx_max]
    node_largest_severity = row_max['cmdb_id']
    node_largest_severity_ts = row_max['earliest_anomaly_iso']
    node_summary = f"Largest severity on {node_largest_severity}; earliest anomaly at {node_largest_severity_ts}"
else:
    node_summary = "No node metrics matched the filters in the window."

# Keep intermediate tables in kernel variables: trace_thresholds, trace_frontend_window, agg_trace_frontend, metric_thresholds, node_window, agg_node
trace_thresholds, trace_frontend_window, trace_top20, trace_self_summary, metric_thresholds, node_window, node_top20, node_summary

```
Out[7]:
```
Summary (plain English, focused):

Quick conclusion
- There is supporting telemetry that links observed anomalies in traces and node metrics around the incident window to the earlier-detected frontend2-0 CPU throttling. The data suggest node-level IO pressure (especially on node-5) beginning at ~2022-03-21T14:30:00+08 often precedes or co-occurs with frontend2-0 trace anomalies and the CPU-throttling run discovered earlier (~14:35), so node resource contention (IO) is a plausible contributing factor. This is correlation (evidence to investigate), not definitive proof.

Details — frontend2-0 traces
- Global thresholds were computed per (cmdb_id, trace_name).
- Windowed traces for frontend2-0 (1647844200–1647846000):
  - Many frontend2-0 trace.* items show anomalies. Notably trace.self.* metrics were flagged: trace.self.error_rate and other trace.self.* entries have anomaly_count > 0.
  - Earliest anomaly among trace.self.* occurred at 2022-03-21T14:30:00+08:00.
  - Several downstream trace metrics (to_* and from_*) also had anomalous counts in the window (some with all window points flagged when thresholds were zero).
- One-line trace summary: "trace.self.* metrics for frontend2-0 have anomalies; earliest at 2022-03-21T14:30:00+08:00."

Details — node-level metrics (node-5 vs node-6)
- Global thresholds computed per (cmdb_id, kpi_name).
- Filtered node metrics (node-5, node-6) for resource-related KPIs in the window show strong anomalies, dominated by IO-related KPIs on node-5:
  - node-5 top anomaly rows include system.io.rkb_s, system.io.avg_q_sz, system.io.r_s, system.cpu.iowait, system.io.util, system.io.r_await, etc., with very large severities (examples: system.io.rkb_s severity ~10239, avg_q_sz ~1140, r_s ~240). Earliest anomaly timestamps for many node-5 IO metrics: 2022-03-21T14:30:00+08:00.
  - node-6 shows elevated load metrics (e.g., system.load.1 severity ~1.56) and increased IO read metrics, but the largest severities (by far) are on node-5.
- One-line node summary: "Largest severity on node-5; earliest anomaly at 2022-03-21T14:30:00+08:00."

Interpretation and recommended next steps
- Temporal correlation:
  - Node-5 IO anomalies begin at ~14:30.
  - Frontend2-0 trace anomalies begin at ~14:30.
  - Frontend2-0 CPU throttling run detected earlier in metric analysis started ~14:35.
  - This timing is consistent with node IO pressure (or some node-level resource contention) degrading service behavior and then triggering CPU throttling on frontend2-0 shortly afterwards.
- Actions to confirm root cause:
  - Verify pod-to-node mapping: check whether frontend2-0 (or its replica) runs on node-5. If frontend2-0 is on node-5, node IO pressure is a strong candidate root cause.
  - Inspect node-5 host logs, iostat, dmesg, kubelet/CRI logs for IO errors, throttling, or disk/scheduler events around 14:30–14:40.
  - Correlate per-pod container metrics on frontend2-0 (cgroup/cpu throttling counters, container CPU limit) and node-level disk latency/queue depth spikes.
  - Review traces for increased latencies or downstream row_count anomalies after 14:30 for services served by frontend2-0.
- Caveat:
  - Many anomaly flags (especially error_rate == 0) result from zero-valued global thresholds and should be interpreted cautiously. Use the non-zero duration/row_count and node IO metrics (which show large numeric deviations) as stronger signals.

If you want, I can:
- Check pod placement to see if frontend2-0 is indeed on node-5 (if pod-to-node mapping is present), or
- Extract the exact timestamps and time series slices for the top node-5 IO KPIs and frontend2-0 CPU throttling series for side-by-side comparison.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                               trace_name  total_points_full  global_P95  global_P5
0            adservice-0      trace.from_frontend-0.duration_mean                 30    0.000026   0.000016
1            adservice-0       trace.from_frontend-0.duration_p95                 30    0.000059   0.000018
2            adservice-0         trace.from_frontend-0.error_rate                 30    0.000000   0.000000
3            adservice-0          trace.from_frontend-0.row_count                 30   17.550000  12.450000
4            adservice-0      trace.from_frontend-1.duration_mean                 30    0.000025   0.000016
...                  ...                                      ...                ...         ...        ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count                 30   14.000000   6.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean                 30    0.000083   0.000070
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95                 30    0.000093   0.000077
1230  shippingservice2-0        trace.from_frontend2-0.error_rate                 30    0.000000   0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count                 30   32.000000  18.450000

[1232 rows x 5 columns],        timestamp      cmdb_id                                   trace_name       value  total_points_full  global_P95   global_P5  is_anomaly
0     1647844200  frontend2-0                trace.from_root.duration_mean    0.045316                 30    0.050749    0.045038       False
1     1647844200  frontend2-0                 trace.from_root.duration_p95    0.074760                 30    0.084704    0.065632       False
2     1647844200  frontend2-0                   trace.from_root.error_rate    0.000000                 30    0.000000    0.000000        True
3     1647844200  frontend2-0                    trace.from_root.row_count  126.000000                 30  140.650000  115.450000       False
4     1647844200  frontend2-0                     trace.self.duration_mean    0.004562                 30    0.004950    0.004488       False
...          ...          ...                                          ...         ...                ...         ...         ...         ...
1075  1647845940  frontend2-0  trace.to_recommendationservice2-0.row_count   91.000000                 30  101.100000   79.450000       False
1076  1647845940  frontend2-0    trace.to_shippingservice2-0.duration_mean    0.000073                 30    0.000083    0.000070       False
1077  1647845940  frontend2-0     trace.to_shippingservice2-0.duration_p95    0.000085                 30    0.000093    0.000077       False
1078  1647845940  frontend2-0       trace.to_shippingservice2-0.error_rate    0.000000                 30    0.000000    0.000000        True
1079  1647845940  frontend2-0        trace.to_shippingservice2-0.row_count   18.000000                 30   32.000000   18.450000        True

[1080 rows x 8 columns],                                         trace_name  total_points_in_window  global_P95   global_P5  anomaly_count  earliest_anomaly_ts       earliest_anomaly_iso  max_value_in_window  min_value_in_window
0                       trace.from_root.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
1                            trace.self.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
2                 trace.to_adservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
3             trace.to_cartservice2-0.duration_p95                      30    0.001000    0.001000             30           1647844200  2022-03-21T14:30:00+08:00             0.001000             0.001000
4               trace.to_cartservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
5           trace.to_checkoutservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
6           trace.to_currencyservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
7     trace.to_productcatalogservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
8     trace.to_recommendationservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
9           trace.to_shippingservice2-0.error_rate                      30    0.000000    0.000000             30           1647844200  2022-03-21T14:30:00+08:00             0.000000             0.000000
10           trace.to_checkoutservice2-0.row_count                      30    7.000000    3.000000             13           1647844260  2022-03-21T14:31:00+08:00             7.000000             1.000000
11             trace.to_adservice2-0.duration_mean                      30    0.000022    0.000017              7           1647844800  2022-03-21T14:40:00+08:00             0.000038             0.000017
12       trace.to_currencyservice2-0.duration_mean                      30    0.000108    0.000095              6           1647844320  2022-03-21T14:32:00+08:00             0.000114             0.000093
13       trace.to_shippingservice2-0.duration_mean                      30    0.000083    0.000070              6           1647844380  2022-03-21T14:33:00+08:00             0.000085             0.000068
14               trace.to_cartservice2-0.row_count                      30  123.650000  100.000000              5           1647844320  2022-03-21T14:32:00+08:00           127.000000            88.000000
15        trace.to_currencyservice2-0.duration_p95                      30    0.000139    0.000132              5           1647844380  2022-03-21T14:33:00+08:00             0.000150             0.000131
16           trace.to_shippingservice2-0.row_count                      30   32.000000   18.450000              5           1647845640  2022-03-21T14:54:00+08:00            33.000000            14.000000
17  trace.to_recommendationservice2-0.duration_p95                      30    0.003415    0.003062              4           1647844200  2022-03-21T14:30:00+08:00             0.003494             0.003037
18           trace.to_currencyservice2-0.row_count                      30  437.250000  297.250000              4           1647844260  2022-03-21T14:31:00+08:00           455.000000           289.000000
19                       trace.from_root.row_count                      30  140.650000  115.450000              4           1647844320  2022-03-21T14:32:00+08:00           145.000000           109.000000, 'trace.self.* metrics for frontend2-0 have anomalies; earliest at 2022-03-21T14:30:00+08:00',                  cmdb_id                                           kpi_name  total_points_full    global_P95     global_P5
0              adservice                                     app.grpc.count                 61    254.000000    205.000000
1              adservice                                       app.grpc.mrt                 61      0.002828      0.002493
2              adservice                                        app.grpc.rr                 61    100.000000    100.000000
3              adservice                                        app.grpc.sr                 61    100.000000    100.000000
4              adservice                                     app.http.count                 61     24.000000     24.000000
...                  ...                                                ...                ...           ...           ...
5341  shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...                 61     34.000000     15.000000
5342  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                 61  84225.000000  48300.000000
5343  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                 61      0.002100      0.000000
5344  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                 61     26.000000     13.500000
5345  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....                 61   2537.250000   1393.000000

[5346 rows x 5 columns],        timestamp cmdb_id              kpi_name         value           kpi_name_lc  total_points_full    global_P95     global_P5  is_anomaly
0     1647844200  node-5     system.cpu.iowait  0.000000e+00     system.cpu.iowait                 61  6.000000e-02  0.000000e+00        True
1     1647844200  node-5  system.cpu.pct_usage  7.610000e+00  system.cpu.pct_usage                 61  9.500000e+00  6.050000e+00       False
2     1647844200  node-5     system.cpu.system  3.010000e+00     system.cpu.system                 61  3.350000e+00  2.540000e+00       False
3     1647844200  node-5       system.cpu.user  4.600000e+00       system.cpu.user                 61  6.500000e+00  3.500000e+00       False
4     1647844200  node-5      system.disk.free  3.886722e+09      system.disk.free                 61  3.887173e+09  3.884635e+09       False
...          ...     ...                   ...           ...                   ...                ...           ...           ...         ...
1297  1647846000  node-6     system.io.w_await  0.000000e+00     system.io.w_await                 61  1.009000e+01  0.000000e+00        True
1298  1647846000  node-6         system.io.w_s  0.000000e+00         system.io.w_s                 61  1.620000e+02  0.000000e+00        True
1299  1647846000  node-6         system.load.1  3.980000e+00         system.load.1                 61  4.610000e+00  2.800000e-01       False
1300  1647846000  node-6        system.load.15  1.150000e+00        system.load.15                 61  2.720000e+00  7.900000e-01       False
1301  1647846000  node-6         system.load.5  1.710000e+00         system.load.5                 61  3.740000e+00  5.200000e-01       False

[1302 rows x 9 columns],    cmdb_id              kpi_name  total_points_in_window  global_P95  global_P5  anomaly_count  earliest_anomaly_ts       earliest_anomaly_iso  max_value_in_window  min_value_in_window      severity
0   node-5       system.io.rkb_s                      31       12.00       0.00             29         1.647844e+09  2022-03-21T14:30:00+08:00            122872.00                 0.00  10239.333333
1   node-5    system.io.avg_q_sz                      31        0.01       0.00             31         1.647844e+09  2022-03-21T14:30:00+08:00                11.40                 0.00   1140.000000
2   node-5         system.io.r_s                      31        1.50       0.00             30         1.647844e+09  2022-03-21T14:30:00+08:00               361.00                 0.00    240.666667
3   node-5     system.cpu.iowait                      31        0.06       0.00             20         1.647844e+09  2022-03-21T14:30:00+08:00                 5.69                 0.00     94.833333
4   node-5        system.io.util                      31        0.75       0.00             23         1.647844e+09  2022-03-21T14:30:00+08:00                49.40                 0.00     65.866667
5   node-5     system.io.r_await                      31        0.38       0.00             30         1.647844e+09  2022-03-21T14:30:00+08:00                16.07                 0.00     42.289474
6   node-5       system.io.await                      31        0.84       0.00             22         1.647844e+09  2022-03-21T14:30:00+08:00                16.07                 0.00     19.130952
7   node-5       system.io.svctm                      31        0.50       0.00             22         1.647844e+09  2022-03-21T14:30:00+08:00                 1.00                 0.00      2.000000
8   node-6         system.load.1                      31        4.61       0.28              4         1.647844e+09  2022-03-21T14:31:00+08:00                 3.98                 0.18      1.555556
9   node-5         system.io.w_s                      31       21.00       0.00             25         1.647844e+09  2022-03-21T14:30:00+08:00                31.00                 0.00      1.476190
10  node-5  system.cpu.pct_usage                      31        9.50       6.05              4         1.647844e+09  2022-03-21T14:31:00+08:00                12.81                 6.02      1.348421
11  node-5     system.cpu.system                      31        3.35       2.54              4         1.647844e+09  2022-03-21T14:31:00+08:00                 4.22                 2.43      1.259701
12  node-5         system.load.1                      31        1.94       0.26              7         1.647844e+09  2022-03-21T14:33:00+08:00                 2.26                 0.18      1.164948
13  node-6         system.load.5                      31        3.74       0.52              5         1.647845e+09  2022-03-21T14:43:00+08:00                 1.71                 0.45      1.155556
14  node-5         system.load.5                      31        1.32       0.49              9         1.647845e+09  2022-03-21T14:36:00+08:00                 1.52                 0.44      1.151515
15  node-6       system.io.svctm                      31        1.72       0.00             18         1.647844e+09  2022-03-21T14:32:00+08:00                 1.87                 0.00      1.087209
16  node-5     system.io.w_await                      31        0.82       0.00             25         1.647844e+09  2022-03-21T14:30:00+08:00                 0.87                 0.00      1.060976
17  node-5        system.load.15                      31        1.02       0.69              9         1.647845e+09  2022-03-21T14:38:00+08:00                 1.07                 0.65      1.049020
18  node-6         system.io.r_s                      31      141.50       0.00             30         1.647844e+09  2022-03-21T14:30:00+08:00               146.50                 0.00      1.035336
19  node-6       system.io.rkb_s                      31   124054.00       0.00             30         1.647844e+09  2022-03-21T14:30:00+08:00            128126.00                 0.00      1.032824, 'Largest severity on node-5; earliest anomaly at 2022-03-21T14:30:00+08:00')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import re
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel (assumed loaded)
# 1) Filter rows for frontend2-0 and list distinct kpi_name values (limit 200) and count
frontend_mask = metric_df['cmdb_id'] == 'frontend2-0'
frontend_kpis_series = pd.Series(metric_df.loc[frontend_mask, 'kpi_name'].dropna().astype(str).unique())
frontend_kpis_df = frontend_kpis_series.head(200).to_frame(name='kpi_name').reset_index(drop=True)
frontend_kpi_count = int(frontend_kpis_series.shape[0])

# 2) Extract node references from kpi_name strings (pattern 'node-<number>' or 'container.node-<number>')
# Normalize matches to 'node-<number>'
def extract_node_ids(kpi_name):
    if pd.isna(kpi_name):
        return []
    s = kpi_name.lower()
    matches = re.findall(r'(?:container\.node-|node-)(\d+)', s)
    return [f'node-{m}' for m in matches]

# Build mapping node_id -> set of kpi_names referencing it
node_map = {}
for kpi in frontend_kpis_series:
    node_ids = extract_node_ids(kpi)
    for nid in set(node_ids):
        node_map.setdefault(nid, set()).add(kpi)

# Prepare DataFrame summarizing counts per node id
if node_map:
    node_counts_df = (
        pd.DataFrame([
            {'node_id': nid, 'referencing_kpi_count': len(kpis), 'sample_kpi': next(iter(kpis))}
            for nid, kpis in node_map.items()
        ])
        .sort_values(by='referencing_kpi_count', ascending=False)
        .reset_index(drop=True)
    )
else:
    node_counts_df = pd.DataFrame(columns=['node_id','referencing_kpi_count','sample_kpi'])

# 3) Top 10 metric rows for frontend2-0 inside the incident window, sorted by timestamp asc
start_ts = 1647844200
end_ts = 1647846000
frontend_window = metric_df[
    (metric_df['cmdb_id'] == 'frontend2-0') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

frontend_window_top10 = frontend_window.sort_values('timestamp', ascending=True).loc[:, ['timestamp','cmdb_id','kpi_name','value']].head(10).reset_index(drop=True)

# Display compact results (DataFrames and count). Kernel variables kept: frontend_kpis_df, frontend_kpi_count, node_counts_df, frontend_window_top10
frontend_kpis_df, frontend_kpi_count, node_counts_df, frontend_window_top10

```
Out[8]:
```
```python
# Summarize frontend2-0 KPI names, node references, and top-10 window rows (compact, plain English)
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse variables from kernel:
# frontend_kpis_df (up to first 200 unique kpi_name for frontend2-0),
# frontend_kpi_count (integer count of distinct kpi_names),
# node_counts_df (node_id, referencing_kpi_count, sample_kpi),
# frontend_window_top10 (top 10 rows in the incident window for frontend2-0)

# Prepare compact samples
sample_kpis = list(frontend_kpis_df['kpi_name'].head(20))
node_summary_rows = []
if not node_counts_df.empty:
    for _, r in node_counts_df.iterrows():
        node_summary_rows.append(f"{r['node_id']}: {int(r['referencing_kpi_count'])} kpi_names")
else:
    node_summary_rows = ["(no node references found)"]

# Format top-10 window rows with ISO timestamps
def to_iso(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).isoformat()

top10_lines = []
for _, r in frontend_window_top10.iterrows():
    iso = to_iso(r['timestamp'])
    top10_lines.append(f"{iso} | {r['kpi_name']} = {r['value']}")

summary_text = (
    "Summary for frontend2-0 (from metric.csv):\n\n"
    f"1) Distinct kpi_name count: {int(frontend_kpi_count)} (showing up to first 200; sample up to 20 below)\n"
    f"- Sample kpi_names (first 20):\n  " + "\n  ".join(sample_kpis) + "\n\n"
    f"2) Node references extracted from kpi_name strings (pattern node-<n> or container.node-<n>):\n"
    + "  " + "; ".join(node_summary_rows) + "\n\n"
    f"3) Top 10 metric rows for frontend2-0 inside incident window (sorted by timestamp asc):\n"
    + "  " + "\n  ".join(top10_lines) + "\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                              kpi_name
0           container.node-6.container_cpu_cfs_periods
1    container.node-6.container_cpu_cfs_throttled_p...
2    container.node-6.container_cpu_cfs_throttled_s...
3      container.node-6.container_cpu_load_average_10s
4        container.node-6.container_cpu_system_seconds
..                                                 ...
195  mesh.source.frontend2.shippingservice2.istio_r...
196  mesh.source.frontend2.shippingservice2.istio_r...
197  mesh.source.frontend2.shippingservice2.istio_r...
198  mesh.source.frontend2.unknown.istio_request_by...
199  mesh.source.frontend2.unknown.istio_request_du...

[200 rows x 1 columns], 204,   node_id  referencing_kpi_count                                         sample_kpi
0  node-6                     64  container.node-6.container_fs_io_time_weighted...,     timestamp      cmdb_id                                           kpi_name       value
0  1647844200  frontend2-0         container.node-6.container_cpu_cfs_periods  301.000000
1  1647844200  frontend2-0  container.node-6.container_cpu_cfs_throttled_p...    0.000000
2  1647844200  frontend2-0  container.node-6.container_cpu_cfs_throttled_s...    0.030265
3  1647844200  frontend2-0    container.node-6.container_cpu_load_average_10s    0.000000
4  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    0.000000
5  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    0.000000
6  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    6.469250
7  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    0.166525
8  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    0.000000
9  1647844200  frontend2-0  mesh.destination.unknown.frontend2.istio_reque...    0.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import re

# Reuse metric_df from kernel
components = ['frontend2-0','adservice-2','adservice-0','checkoutservice-0',
              'checkoutservice-1','checkoutservice-2','node-5','node-6']

# Helper to extract node ids from a kpi_name (returns list like ['node-6', ...])
node_pattern = re.compile(r'(?:container\.node-|node-)(\d+)', flags=re.IGNORECASE)
def extract_node_ids_from_kpi(kpi):
    if pd.isna(kpi):
        return []
    matches = node_pattern.findall(kpi)
    return [f'node-{m}' for m in matches]

rows = []
# Also keep a fuller intermediate table per component (stored but not printed)
component_details = {}

for comp in components:
    comp_rows = metric_df[metric_df['cmdb_id'] == comp].copy()
    unique_kpis = comp_rows['kpi_name'].dropna().astype(str).unique()
    # Find kpis that reference node patterns
    kpis_with_nodes = []
    node_count_map = {}
    for k in unique_kpis:
        ids = extract_node_ids_from_kpi(k)
        if ids:
            kpis_with_nodes.append(k)
            for nid in set(ids):
                node_count_map[nid] = node_count_map.get(nid, 0) + 1
    # Prepare samples and counts
    sample_kpis = kpis_with_nodes[:5]  # up to 5 sample kpis showing node reference
    sample_kpis_str = '; '.join(sample_kpis) if sample_kpis else ''
    # Node references summary top up to 10
    node_refs_sorted = sorted(node_count_map.items(), key=lambda x: x[1], reverse=True)
    node_refs_top = node_refs_sorted[:10]
    node_refs_str = '; '.join([f"{nid} ({cnt})" for nid, cnt in node_refs_top]) if node_refs_top else ''
    # Inferred node logic
    unique_nodes = sorted(node_count_map.keys())
    if len(unique_nodes) == 1:
        inferred = unique_nodes[0]
    elif len(unique_nodes) > 1:
        inferred = 'multiple'
    else:
        inferred = 'none'
    # Keep component detail record
    rows.append({
        'component': comp,
        'inferred_node': inferred,
        'node_references': node_refs_str,
        'sample_kpis': sample_kpis_str,
        'distinct_kpi_count': int(len(unique_kpis)),
        'kpi_names_with_node_refs_count': int(len(kpis_with_nodes))
    })
    # store intermediate detail DataFrame for this component
    component_details[comp] = pd.DataFrame({
        'kpi_name': kpis_with_nodes
    }).head(200)  # limited cached view

summary_df = pd.DataFrame(rows, columns=[
    'component','inferred_node','node_references','sample_kpis',
    'distinct_kpi_count','kpi_names_with_node_refs_count'
])

# Final compact output (limited to top 20 rows)
summary_df.head(20)

```
Out[9]:
```
```python
# Plain-English compact summary (displayed as a single string)
summary = (
    "Component -> inferred_node mapping (from scanning kpi_name strings for 'node-<n>' patterns):\n\n"
    "- frontend2-0: inferred_node = node-6 (64 kpi_names reference node-6). Total distinct kpi_names: 204.\n"
    "- adservice-2: inferred_node = node-5 (64 kpi_names reference node-5). Total distinct kpi_names: 94.\n"
    "- adservice-0: inferred_node = node-6 (64 kpi_names reference node-6). Total distinct kpi_names: 94.\n"
    "- checkoutservice-0: inferred_node = node-6 (64 kpi_names reference node-6). Total distinct kpi_names: 144.\n"
    "- checkoutservice-1: inferred_node = node-6 (64 kpi_names reference node-6). Total distinct kpi_names: 148.\n"
    "- checkoutservice-2: inferred_node = node-5 (64 kpi_names reference node-5). Total distinct kpi_names: 144.\n"
    "- node-5: inferred_node = none (no kpi_names reference another node). Total distinct kpi_names: 58.\n"
    "- node-6: inferred_node = none (no kpi_names reference another node). Total distinct kpi_names: 59.\n\n"
    "Notes:\n"
    "- For each application/pod component above, exactly one node id (node-5 or node-6) is referenced by its kpi_name strings; we report that node as the inferred host.\n"
    "- Each referenced node id appears in ~64 kpi_name entries for those components, indicating many container-scoped KPIs include the container.node-<n> token.\n"
    "- Nodes themselves (node-5, node-6) do not reference other nodes in their kpi_names.\n"
    "- Sample KPI name patterns observed (examples): container.node-6.container_cpu_cfs_periods; container.node-6.container_cpu_cfs_throttled_...; container.node-5.container_fs_reads./dev/vda.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component inferred_node node_references                                        sample_kpis  distinct_kpi_count  kpi_names_with_node_refs_count
0        frontend2-0        node-6     node-6 (64)  container.node-6.container_cpu_cfs_periods; co...                 204                              64
1        adservice-2        node-5     node-5 (64)  container.node-5.container_cpu_cfs_periods; co...                  94                              64
2        adservice-0        node-6     node-6 (64)  container.node-6.container_cpu_cfs_periods; co...                  94                              64
3  checkoutservice-0        node-6     node-6 (64)  container.node-6.container_cpu_cfs_periods; co...                 144                              64
4  checkoutservice-1        node-6     node-6 (64)  container.node-6.container_cpu_cfs_periods; co...                 148                              64
5  checkoutservice-2        node-5     node-5 (64)  container.node-5.container_cpu_cfs_periods; co...                 144                              64
6             node-5          none                                                                                     58                               0
7             node-6          none                                                                                     59                               0```
```